# S1–S7 · Full pipeline in one notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s1_s7_pipeline.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s1_s7_pipeline.ipynb)

The week-by-week notebooks, merged in order so that one *Run all* goes from the raw data to every verdict. Each stage
is the corresponding notebook unchanged, below its own title; they share the setup cell and the parameters cell.

| Stage | Notebook it reproduces | Needs |
|---|---|---|
| S1 | `s1_harness` · data integrity, frozen split, HLUT, projector, range diagnostics | CPU |
| S2 | `s2_synthetic_benchmark` · theory checks, identities on real patients | CPU |
| S3 | `s3_real_diagnostic` · L1-HU and L2-SPR models, beam spectra, go/no-go 1 | GPU |
| S4–S5 | `s4_regression_arms` · arms R1–R6 over three seeds, pre-registered verdict | GPU |
| S6 | `s6_dosimetry` · proton and photon plans, proxy validation, go/no-go 2 | CPU (hours), pyRadPlan |
| S7–S8 | `s7_generative` · stochastic-interpolant generator, projected CRPS, comparison with R3 and R6 | GPU |

Every stage runs on the real SynthRAD2025 data (Hugging Face mirror `developper669/synthrad2025`, tag `v2`) and writes
to its own `outputs/s<k>/` folder, exactly as the separate notebooks do, so the two can be mixed: a stage interrupted
here can be finished in its own notebook and vice versa. `s0_zenodo_to_huggingface` (one-time mirror) is not part of
the pipeline.

**Automatic resumption.** After a crash, an error or an expired session (Colab, Kaggle), reopen the notebook and
*Run all* again with the same parameters; nothing has to be cleaned up.

- Stages that finished are skipped: their cells run as `pass` and print where their report is
  (`outputs/pipeline_state.json`, kept on Drive on Colab and in `/kaggle/working` on Kaggle; on Kaggle, attach the
  previous version's output as an input). A stage runs again when its parameters change, when a stage it depends on
  finished again, when its report was deleted, or when it is listed in `FORCE_STAGES`.
- A stage that stopped early (an error, no GPU, upstream runs missing) runs again and continues where it stopped:
  fold checkpoints are reused, a fold cut mid-training resumes from its last save (every 1000 steps, exactly the
  same model as an uninterrupted run), per-case evaluations, dosimetry cases and generator samples are cached.
- The dataset is fetched again on a new Colab or Kaggle session (resumable, with retries on network errors); the
  volume cache is rebuilt from it.


### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

# OpenBLAS 0.3.27 can corrupt multithreaded matrix products, and pyRadPlan (S6) optimises plans and computes dose with
# NumPy: one BLAS thread per process. The variable only acts before NumPy is loaded; threadpoolctl covers a later cell.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if "numpy" in sys.modules:
    try:
        from threadpoolctl import threadpool_limits

        BLAS_LIMIT = threadpool_limits(1, user_api="blas")
    except ImportError:
        print("NumPy was loaded before this cell: restart the kernel so that OPENBLAS_NUM_THREADS=1 applies")

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():  # an earlier, interrupted attempt or another folder
        if any(destination.iterdir()):
            raise RuntimeError(f"{destination} exists and is not a clone of {REPO_URL}; remove it or set SCTRANGE_HOME")
        destination.rmdir()
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables of
the same meaning, given in brackets, override them for unattended runs (`jupyter nbconvert --execute`). Pre-registered
verdicts assume the defaults (30 000 steps, 5 folds, seeds 0, 1, 2, 20 dosimetry cases, the S7 budget); smaller values
give a quick pass only. S1 freezes the local test split the first time it runs on both tasks.

| Platform | Dataset, cache | Outputs | GPU |
|---|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` | every visible GPU |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) | *Runtime → Change runtime type → T4 GPU* |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` | *Settings → Accelerator → GPU T4 × 2*, *Internet on* |


In [ ]:
# @title Parameters { display-mode: "form" }
# shared
TASK = "1"  # @param ["1", "2"]  S2-S7: 1 = MRI, 2 = CBCT [SCT_TASK]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only
FORCE_STAGES = ""  # @param {type:"string"}  stages to run again even if complete, e.g. "S3,S4" [SCT_FORCE_STAGES]
# S1
TASKS = "1,2"  # @param ["1,2", "1", "2"]  tasks checked by S1 [SCT_TASKS]
S1_MAX_CASES = 0  # @param {type:"integer"}  S1 diagnostics, 0 = every development case [SCT_MAX_CASES]
# S2
S2_CASES = 20  # @param {type:"integer"}  S2 section 7 cases, 0 = all [SCT_MAX_CASES]
# S3, S4, S7 training
TRAIN = True  # @param {type:"boolean"}  train missing folds here (needs a GPU) [SCT_TRAIN_IN_NOTEBOOK]
STEPS = 30000  # @param {type:"integer"}  optimisation steps per fold [SCT_STEPS]
FOLDS = 5  # @param {type:"integer"}  cross-fitting folds [SCT_FOLDS]
SEED_LIST = "0,1,2"  # @param {type:"string"}  S4 seeds, also recomputed by S6 [SCT_SEEDS]
# S6
DOSE_CASES = 20  # @param {type:"integer"}  dosimetry cases [SCT_DOSE_CASES]
WORKERS = 1  # @param {type:"integer"}  cases dosed in parallel [SCT_WORKERS]
# S7
ETAS = "0,0.05,0.1,0.2"  # @param {type:"string"}  [SCT_ETAS]
SWEEP_CASES = 30  # @param {type:"integer"}  [SCT_SWEEP_CASES]
STEP_CURVE_CASES = 10  # @param {type:"integer"}  [SCT_STEP_CASES]
MAIN_CASES = 100  # @param {type:"integer"}  [SCT_MAIN_CASES]


### Resumption

The state of every stage is read from the outputs; the table below shows what the previous runs left. Stages marked
`complete` with unchanged inputs are skipped below.


In [ ]:
from sctrange.pipeline import StageTracker
from sctrange.runtime import resolve_paths

PIPELINE_OUTPUTS = resolve_paths(REPO_ROOT, USE_DRIVE).output_root
PIPELINE = StageTracker(PIPELINE_OUTPUTS, os.environ.get("SCT_FORCE_STAGES", FORCE_STAGES).split(","))
PIPELINE.install()
PIPELINE.show()


# S1 · Range-aware evaluation harness

This notebook validates every week-1 component and produces the artefacts that later weeks depend on:

1. a reproducible environment record;
2. a frozen, hash-verified local test split;
3. an HU → SPR conversion identical to the one used by the SynthRAD2025 dose engine (pyRadPlan), checked on the HU of a
   real CT;
4. a differentiable water-equivalent-thickness (WET) projector, checked against pyRadPlan's Siddon ray tracer and
   finite differences on real patient anatomy, and against exact chord lengths on disks;
5. per-case range diagnostics on the development pool.

All logic lives in the `sctrange` package; the notebook configures, runs and reports. Every check uses the real
data except the two analytic geometry checks (disk chords, target segments), which need a shape whose exact
path lengths are known; the same projector is then compared with the Siddon tracer on a patient.
The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead). `SCT_TASKS` selects the tasks (default `1,2`).
`SCT_PREDICTION_ROOT` optionally points to a folder of `<case_id>.mha` predictions to diagnose next to the
bulk-density baseline; `SCT_MAX_CASES` limits the diagnostics to the first cases (they take a few seconds per case on
a CPU). pyRadPlan is optional: without it the two dose-engine comparisons are reported as not run.

*Stage notebook: `notebooks/s1_harness.ipynb`.*


In [ ]:
PIPELINE.begin("S1", {'tasks': TASKS, 'max_cases': S1_MAX_CASES}, PIPELINE_OUTPUTS / "s1" / "s1_report.json", after=())


In [ ]:
import importlib.util
import json
import os
import platform
import subprocess
import warnings
from importlib import metadata

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from IPython.display import display

from sctrange.data import bulk_density_sct, central_slab, discover_cases, load_case, read_prediction, read_volume
from sctrange.download import ensure_dataset, validate_dataset
from sctrange.runs import cached_result, config_key, files_signature
from sctrange.evaluation import DiagnosticConfig, evaluate_case, prepare_case
from sctrange.hlut import HLUT
from sctrange.metrics import range_error
from sctrange.training import visible_devices
from sctrange.runtime import frozen_split_path, resolve_paths, save_split_copy
from sctrange.plotting import apply_style, plot_case, plot_hlut, plot_mae_vs_range, plot_projector_checks
from sctrange.split import SplitSpec, load_split, select_cases, split_hash, stratified_split, write_split
from sctrange.validation import (
    benchmark_range_loss,
    disk_chord_errors,
    gradcheck_range_loss,
    siddon_relative_errors,
    siddon_relative_errors_case,
    target_path_errors,
)

PYRADPLAN = importlib.util.find_spec("pyRadPlan") is not None

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASKS = tuple(os.environ.get("SCT_TASKS", TASKS).split(","))
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
S1_MAX_CASES = int(os.environ.get("SCT_MAX_CASES", S1_MAX_CASES)) or None
DATA_ROOT = PATHS.data_root
PREDICTION_ROOT = Path(os.environ["SCT_PREDICTION_ROOT"]) if "SCT_PREDICTION_ROOT" in os.environ else None
OUTPUT_DIR = PATHS.output_root / "s1"
FIGURE_DIR = OUTPUT_DIR / "figures"
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)

SEED = 20261013
TEST_FRACTION = 0.2
DEVICES = visible_devices()  # ["cuda:0", "cuda:1"] on Kaggle T4 x 2
DEVICE = DEVICES[0]
VALIDATION_ANGLES = (0.0, 17.0, 45.0, 90.0, 133.0, 210.0, 301.0)
SIDDON_ANGLES = (0.0, 60.0, 135.0)
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())

In [ ]:
DATA_ROOT = ensure_dataset(DATA_ROOT, TASKS, download=DOWNLOAD)
SPLIT_PATH = FROZEN_SPLIT_PATH


def shown(path: Path | None) -> str:
    """Repository-relative path when possible, so that outputs carry no machine-specific prefix."""
    if path is None:
        return "none"
    try:
        return path.resolve().relative_to(REPO_ROOT.resolve()).as_posix()
    except ValueError:
        return str(path)


print(f"data       : {shown(DATA_ROOT)} (tasks {', '.join(TASKS)})")
print(f"predictions: {shown(PREDICTION_ROOT)}")
print(f"split      : {shown(SPLIT_PATH)}")
print(f"device     : {DEVICE}")
print(f"pyRadPlan  : {'installed' if PYRADPLAN else 'not installed (engine comparisons skipped)'}")

### 0 · Dataset integrity

Every case is checked before use: `ct`, `mask` and the source image present, identical geometry, isotropic in-plane
spacing, a non-empty body mask and CT values inside [-1024, 3071] HU (the clipping of the v2 release). On the real
dataset the result comes with the data in `<root>/manifest.csv` (published with the Hugging Face
mirror); delete it to re-check locally.

In [ ]:
manifest_path = DATA_ROOT / "manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype={"task": str, "case_id": str}, keep_default_na=False)
    manifest["ok"] = manifest["ok"].astype(str).str.lower() == "true"
else:
    manifest = validate_dataset(DATA_ROOT, TASKS)
manifest = manifest[manifest["task"].isin(TASKS)]
INVALID_CASES = set(manifest.loc[~manifest["ok"], "case_id"])

display(manifest.groupby(["task", "region", "center"])["ok"].agg(cases="size", valid="sum"))
if INVALID_CASES:
    display(manifest.loc[~manifest["ok"], ["case_id", "problems"]])
print(f"{int(manifest['ok'].sum())}/{len(manifest)} cases pass; invalid cases are left out of the diagnostics")

### 1 · Frozen local test split

SynthRAD2025 keeps its test ground truth private, so model selection needs a local hold-out drawn from the public training cases. The split is stratified by task, region and centre and uses an independent seed per stratum, so adding cases to one stratum never moves cases in another. It is written once with its SHA-256, committed, and only verified afterwards: `load_split` refuses a modified file.

On the real dataset the canonical file is `splits/synthrad2025_local_test.json`. It can also be produced from the command line and committed:

```bash
sctrange-split --data-root "$SYNTHRAD2025_ROOT" --out splits/synthrad2025_local_test.json
git add splits/ && git commit -m "Freeze SynthRAD2025 local test split"
```

The first run on the real data creates the file; commit it then (`splits/synthrad2025_local_test.json` and its
`.sha256`). Every later run, and every later notebook, only verifies it.

The first valid development case is the *reference case* of the checks on real anatomy below; test cases are
never loaded.

In [ ]:
records = [r for r in discover_cases(DATA_ROOT) if r.task in TASKS]
available = {record.case_id for record in records}

if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    split_status = "verified existing split"
else:
    if set(TASKS) != {"1", "2"}:
        raise RuntimeError("the frozen split is created once over both tasks: run with TASKS = \"1,2\" the first time")
    split = stratified_split(records, SplitSpec(test_fraction=TEST_FRACTION, seed=SEED))
    write_split(split, SPLIT_PATH)
    split = load_split(SPLIT_PATH)
    split_status = "created and verified"
    print(f"frozen split created: commit {SPLIT_PATH.name} and its .sha256 (git add splits/)")
copy = save_split_copy(SPLIT_PATH, PATHS)
if copy is not None:
    print(f"copy kept with the outputs: {copy}")

assigned = set(split["test"]) | set(split["development"])
split_summary = {
    "status": split_status,
    "sha256": split_hash(split),
    "test cases": len(split["test"]),
    "development cases": len(split["development"]),
    "cases on disk not in split": len(available - assigned),
    "split cases missing on disk": len({c for c in assigned if c[0] in TASKS} - available),
}
assert not set(split["test"]) & set(split["development"])

display(pd.Series(split_summary, name="value").to_frame())
display(pd.DataFrame(split["strata"]).T.rename_axis("stratum"))

In [ ]:
development = [record for record in records
               if record.case_id in set(split["development"]) and record.case_id not in INVALID_CASES]
if S1_MAX_CASES:  # a region-stratified subset, not the first case ids (which all come from one region and centre)
    keep = set(select_cases(development, S1_MAX_CASES, SEED))
    development = [record for record in development if record.case_id in keep]
if not development:
    raise RuntimeError(f"no valid development case of task(s) {', '.join(TASKS)} under {DATA_ROOT}")
REFERENCE = development[0]
volumes = load_case(REFERENCE, ("ct", "mask"))
REF_CT, REF_MASK, REF_SPACING = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
print(f"reference case: {REFERENCE.case_id} ({REFERENCE.region}, centre {REFERENCE.center}), "
      f"{REF_CT.shape[0]} slices of {REF_CT.shape[1]} × {REF_CT.shape[2]}, spacing {tuple(round(s, 2) for s in REF_SPACING)} mm")

### 2 · Environment

Versions and the code revision are stored with every result so that any number can be traced back to the exact software that produced it.

In [ ]:
def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"


def git_revision(root: Path) -> str:
    try:
        revision = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=root, capture_output=True,
                                  text=True, check=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=root, capture_output=True, text=True).stdout
        return revision + ("-dirty" if dirty.strip() else "")
    except (OSError, subprocess.CalledProcessError):
        return "no git repository"


environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": DEVICE,
    **{name: package_version(name) for name in ("numpy", "scipy", "SimpleITK", "pandas", "pyradplan", "sctrange")},
    "git_revision": git_revision(REPO_ROOT),
    "dataset": f"SynthRAD2025 training v2, tasks {', '.join(TASKS)}",
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
display(pd.Series(environment, name="value").to_frame())

### 3 · HU → SPR conversion

The range proxy must convert HU exactly as the dose engine does. The SynthRAD2025 dose evaluation uses pyRadPlan (v0.2.8), whose default table is identical in v0.5.0 and is applied with `np.interp`: linear interpolation, constant beyond the end nodes.

Checks: the bundled table equals the installed one; the NumPy and Torch paths reproduce `CT.compute_wet` on the HU of four slices of the reference CT; the curve is monotone with water at 1.

This is the engine's generic table, not a scanner-specific stoichiometric calibration. It is the right choice for consistency with the challenge dosimetry; if a calibrated curve is adopted, it must replace the table in both the proxy and the dose engine.

In [ ]:
hlut = HLUT.pyradplan_default()
hlut_checks = {
    "fingerprint": hlut.fingerprint(),
    "monotone": bool(np.all(np.diff(hlut.numpy(np.linspace(-2000, 4000, 6001))) >= 0)),
    "SPR at 0 HU": float(hlut.numpy(np.array(0.0))),
}
if PYRADPLAN:
    from pyRadPlan.ct import CT

    installed = HLUT.from_pyradplan()
    probe = np.ascontiguousarray(REF_CT[central_slab(REF_MASK, 4)], dtype=np.float32)
    image = sitk.GetImageFromArray(probe)
    image.SetSpacing(tuple(float(s) for s in REF_SPACING))
    engine = sitk.GetArrayFromImage(CT(cube_hu=image).compute_wet(np.stack([hlut.hu, hlut.spr], axis=1)))
    hlut_checks.update({
        "matches installed pyRadPlan": bool(np.array_equal(installed.hu, hlut.hu) and np.array_equal(installed.spr, hlut.spr)),
        "max |numpy - engine|": float(np.abs(hlut.numpy(probe) - engine).max()),
        "max |torch - engine|": float(np.abs(hlut.torch(torch.from_numpy(probe).double()).numpy() - engine).max()),
        "HU values compared": f"{probe.size} ({REFERENCE.case_id}, {probe.min():.0f} to {probe.max():.0f} HU)",
    })
    assert hlut_checks["matches installed pyRadPlan"]
    assert hlut_checks["max |numpy - engine|"] < 1e-12 and hlut_checks["max |torch - engine|"] < 1e-12
else:
    print("pyRadPlan not installed: the bundled table is used without the engine comparison (pip install -e '.[dose]')")
assert hlut_checks["monotone"] and hlut_checks["SPR at 0 HU"] == 1.0

display(pd.Series(hlut_checks, name="value").to_frame())
display(pd.DataFrame({"HU": hlut.hu, "SPR": hlut.spr, "slope to next node (1/HU)": np.append(hlut.slopes(), np.nan)}))
plot_hlut(hlut).savefig(FIGURE_DIR / "hlut.png", bbox_inches="tight")
plt.show()

### 4 · Differentiable WET projector

Each axial slice is padded to a square canvas and rotated with `grid_sample` so that the beam travels along the row axis; a masked sum over rows gives one integral per ray, in millimetres of water. Segments run from beam entry either to the body exit (*chord*) or to the distal edge of the target (*target*), the latter being what governs the range of a proton stopping in the target.

Five independent checks, the last two on the reference patient:

| Check | Reference | Criterion |
|---|---|---|
| Disk chords | exact chord length `2√(R² − u²)` | median < 0.1 mm, max < 0.5 mm |
| Target segments | `√(R² − u²) + √(r² − u²)` | max < 1.5 mm (one pixel at the distal edge) |
| Heterogeneous phantom | pyRadPlan Siddon tracer (water, lung and bone inserts) | median < 1 %, max < 4 % |
| Real anatomy | pyRadPlan Siddon tracer on the largest slice of the reference CT, rays crossing ≥ 20 mm WET | median < 1 % |
| Gradient | finite differences in float64 on a 17 × 19 patch of the reference CT | `torch.autograd.gradcheck` passes |

In [ ]:
chord = disk_chord_errors(VALIDATION_ANGLES)
target = target_path_errors(VALIDATION_ANGLES)
empty = pd.DataFrame(columns=["angle_deg", "offset_px", "siddon_mm", "projector_mm", "relative_error"], dtype=float)
siddon = siddon_relative_errors(hlut, SIDDON_ANGLES) if PYRADPLAN else empty
siddon_real = (siddon_relative_errors_case(hlut, REF_CT, REF_MASK, REF_SPACING[0], SIDDON_ANGLES) if PYRADPLAN
               else empty)

projector_checks = {
    "chord median |error| (mm)": chord["median_abs_mm"].max(),
    "chord max |error| (mm)": chord["max_abs_mm"].max(),
    "target max |error| (mm)": target["max_abs_mm"].max(),
    "target rays valid inside / none outside": bool(target["valid_inside"].all() and not target["valid_outside"].any()),
    "Siddon median relative error": siddon["relative_error"].median() if len(siddon) else float("nan"),
    "Siddon max relative error": siddon["relative_error"].max() if len(siddon) else float("nan"),
    "Siddon rays compared": len(siddon),
    "real anatomy Siddon median relative error": siddon_real["relative_error"].median() if len(siddon_real) else float("nan"),
    "real anatomy Siddon p95 relative error": siddon_real["relative_error"].quantile(0.95) if len(siddon_real) else float("nan"),
    "real anatomy Siddon rays compared": len(siddon_real),
    "gradcheck": gradcheck_range_loss(hlut, SEED, REF_CT, REF_MASK),
}
assert projector_checks["chord median |error| (mm)"] < 0.1 and projector_checks["chord max |error| (mm)"] < 0.5
assert projector_checks["target max |error| (mm)"] < 1.5 and projector_checks["target rays valid inside / none outside"]
if PYRADPLAN:
    assert projector_checks["Siddon median relative error"] < 0.01 and projector_checks["Siddon max relative error"] < 0.04
assert projector_checks["gradcheck"]

display(pd.Series(projector_checks, name="value").to_frame())
display(chord.merge(target, on="angle_deg", suffixes=(" chord", " target")).round(3))
if len(siddon_real):
    display(siddon_real.groupby("angle_deg")[["siddon_mm", "relative_error"]]
            .agg(["median", "max"]).round(4).rename_axis(f"real anatomy, {REFERENCE.case_id}"))
plot_projector_checks(chord, target, siddon).savefig(FIGURE_DIR / "projector_checks.png", bbox_inches="tight")
plt.show()

Cost of one range-loss forward and backward pass on the reference case (16 slices around its largest body section, cropped to the body as in training, three beams), with the bulk-density baseline as sCT, on the active device.

In [ ]:
benchmark = benchmark_range_loss(hlut, REF_CT, REF_MASK, bulk_density_sct(REF_MASK), REF_SPACING[0],
                                 (0.0, 120.0, 240.0), device=DEVICE)
display(pd.Series({"case": REFERENCE.case_id, **benchmark}, name="value").to_frame())

### 5 · Range diagnostics on the development pool

Test cases are never loaded here. Each result is cached in `outputs/s1/cache_eval/`, so a rerun after an interrupted session only evaluates the cases that are missing. For each development case, the bulk-density baseline (water inside the body, as in the challenge) and, when available, a prediction are compared with the reference CT. SynthRAD2025 releases no PTV, so a 25 mm sphere at the body centroid serves as a surrogate target defining the distal edge.

Per case and source:

- `mae_hu`: mean absolute HU error in the body;
- `*_rms_mm`, `*_p95_mm`, `*_cvar_mm`: RMS, 95th percentile and upper-decile mean of |ΔWET| over rays, for chords and target segments;
- `*_coherence`: median of |Σε| / Σ|ε| along rays, close to 1/√n for white errors and to 1 for errors that accumulate;
- `*_intra_ratio`, `*_global_ratio`: RMS ΔWET divided by that of the same errors shuffled within tissue classes, or globally. Both shuffles leave the MAE unchanged, so a ratio away from 1 shows that range error depends on the spatial layout of errors and not only on their magnitude.

In [ ]:
prediction_label = "prediction"



def diagnose(record, path):
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sct = bulk_density_sct(mask) if path is None else read_prediction(path, ct.shape)
    return evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)


rows = []
for index, record in enumerate(development, 1):
    paths = {"bulk density": None}
    if PREDICTION_ROOT is not None and (PREDICTION_ROOT / f"{record.case_id}.mha").exists():
        paths[prediction_label] = PREDICTION_ROOT / f"{record.case_id}.mha"
    data = files_signature([record.path("ct"), record.path("mask")], mtime=False)
    for source, path in paths.items():
        key = {"config": config_key(DIAGNOSTIC), "hlut": hlut.fingerprint(), "data": data,
               "prediction": files_signature([path]) if path else None}
        metrics = cached_result(OUTPUT_DIR / "cache_eval" / f"{record.case_id}_{source.replace(' ', '_')}.pkl", key,
                                lambda: diagnose(record, path))
        rows.append({"case_id": record.case_id, "region": record.region, "center": record.center,
                     "source": source, **metrics})
    if index % 50 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

diagnostics = pd.DataFrame(rows)
diagnostics.to_csv(OUTPUT_DIR / "range_diagnostics.csv", index=False)
print(f"{diagnostics['case_id'].nunique()} development cases, {len(diagnostics)} evaluations")

columns = ["mae_hu", "chord_rms_mm", "target_rms_mm", "target_cvar_mm", "target_coherence",
           "target_intra_ratio", "target_global_ratio"]
display(diagnostics.groupby(["source", "region"])[columns].mean().round(3))

In [ ]:
plot_mae_vs_range(diagnostics).savefig(FIGURE_DIR / "mae_vs_range.png", bbox_inches="tight")
plt.show()
display(diagnostics.groupby("source")[["mae_hu", "target_rms_mm", "target_coherence"]]
        .corr(method="spearman").round(3))

One development case in detail: reference slice, signed HU error, and signed ΔWET per ray to the distal target edge for the first three beams. The slice shown is the one with the most target rays.

In [ ]:
example = next((row for row in rows if row["source"] == prediction_label), rows[0])
record = next(r for r in development if r.case_id == example["case_id"])
volumes = load_case(record, ("ct", "mask"))
sct = (read_volume(PREDICTION_ROOT / f"{record.case_id}.mha").array if example["source"] == prediction_label
       else bulk_density_sct(volumes["mask"].array))
case = prepare_case(volumes["ct"].array, volumes["mask"].array, sct, volumes["ct"].spacing, DIAGNOSTIC)
result = range_error(case.sct, case.ct, case.mask, case.projector, hlut, case.target)
slice_index = int(result.valid.sum(dim=(0, 2)).argmax())

plot_case(case.ct.cpu().numpy(), case.sct.cpu().numpy(), case.mask.cpu().numpy(),
          result.delta_mm.cpu().numpy(), result.valid.cpu().numpy(), DIAGNOSTIC.angles_deg,
          slice_index, f"{record.case_id} · {example['source']}").savefig(FIGURE_DIR / "example_case.png", bbox_inches="tight")
plt.show()

### 6 · S1 status

Every item is evaluated from the results above and written to `outputs/s1/s1_report.json` with the environment record.

In [ ]:
status = pd.DataFrame([
    ("pyRadPlan installed", environment["pyradplan"], environment["pyradplan"] != "not installed"),
    ("HU → SPR identical to dose engine", f"fingerprint {hlut.fingerprint()}",
     PYRADPLAN and hlut_checks["max |numpy - engine|"] < 1e-12 and hlut_checks["max |torch - engine|"] < 1e-12),
    ("Projector vs analytic chords", f"max {projector_checks['chord max |error| (mm)']:.3f} mm",
     projector_checks["chord max |error| (mm)"] < 0.5),
    ("Target segments to distal edge", f"max {projector_checks['target max |error| (mm)']:.3f} mm",
     projector_checks["target max |error| (mm)"] < 1.5),
    ("Projector vs pyRadPlan Siddon",
     f"median {projector_checks['Siddon median relative error']:.2%}" if PYRADPLAN else "not run (pyRadPlan missing)",
     PYRADPLAN and projector_checks["Siddon median relative error"] < 0.01),
    ("Dataset integrity", f"{int(manifest['ok'].sum())}/{len(manifest)} cases valid", not INVALID_CASES),
    ("Projector vs Siddon on real anatomy",
     f"median {projector_checks['real anatomy Siddon median relative error']:.2%} · {REFERENCE.case_id}"
     if PYRADPLAN else "not run (pyRadPlan missing)",
     PYRADPLAN and projector_checks["real anatomy Siddon median relative error"] < 0.01),
    ("Gradient check", f"float64 finite differences · {REFERENCE.case_id}", projector_checks["gradcheck"]),
    ("Local test split", f"{split_summary['status']} · sha256 {split_summary['sha256'][:12]}",
     split_summary["split cases missing on disk"] == 0),
    ("Split covers both tasks", shown(SPLIT_PATH),
     {"task1", "task2"} <= {stratum.split("/")[0] for stratum in split["strata"]}),
    ("Range diagnostics", f"{len(diagnostics)} evaluations", len(diagnostics) > 0),
], columns=["item", "detail", "passed"])

report = {"environment": environment, "hlut": hlut_checks, "projector": projector_checks,
          "benchmark": benchmark, "split": split_summary, "reference_case": REFERENCE.case_id,
          "invalid_cases": sorted(INVALID_CASES),
          "status": status.to_dict(orient="records")}
(OUTPUT_DIR / "s1_report.json").write_text(json.dumps(report, indent=2, default=str))
display(status)

In [ ]:
PIPELINE.finish("S1", complete=bool(True), note=str(""))


# S2 · Theory checks: known posterior, then real patients

Week 2 tests the theory of `docs/theory.md` where the truth is known exactly: stationary random fields with an exact
covariance, and a voxel-wise tissue-mixture posterior whose marginals are closed-form and whose dependence is set by a
Gaussian copula. Section 7 then checks, on real SynthRAD2025 patients, the statements that are identities and
therefore hold for any error field, and measures the quantities they involve. Every statement is checked against
an explicit criterion; section 8 collects the verdicts.

| Section | Statement |
|---|---|
| 1 | Propositions 1 and 1′: sharp rearrangement bounds; Proposition 2 and Corollary 2.1: at a fixed MAE, range error is governed by error coherence |
| 2 | Propositions 3 and 3.1: range error is low-pass along the beam; all angles weight errors by 1/\|k\| |
| 3 | Propositions 4 and 5: the Bayes act for range is the SPR mean; L1 creates a signed range bias |
| 4 | Theorem 6: only the mean is consistent under projection |
| 5 | Proposition 7: tails of range error need the joint law; the FZ score elicits VaR and ES |
| 6 | Proposition 8: projected CRPS separates dependence structures that voxel-wise CRPS cannot |
| 7 | Propositions 1, 1′, 2 and 3 on the SPR errors of real synthetic CTs (SynthRAD2025 development cases) |

Units: 1 voxel = 1 mm along the beam; SPR is dimensionless; range errors are in mm of water.

The notebook runs the code of this repository (`src/`), whether or not the package is installed.

**Synthetic where the truth is needed, real everywhere else.** Sections 1–6 compare measured quantities with their
exact theoretical values, which requires the posterior law of the CT given the source image; no patient dataset
provides it. Section 7 runs on the real SynthRAD2025 training data (development cases of the frozen split of S1,
fetched from Hugging Face if missing): the bulk-density baseline and every out-of-fold prediction already produced by S3 or S4.
The statements that need a posterior are tested on patients, without a known truth, in S3 (Proposition 5) and S4.

*Stage notebook: `notebooks/s2_synthetic_benchmark.ipynb`.*


In [ ]:
PIPELINE.begin("S2", {'task': TASK, 'cases': S2_CASES}, PIPELINE_OUTPUTS / "s2" / "s2_report.json", after=('S1',))


In [ ]:
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sctrange.benchmark import (
    angle_energy_experiment,
    autocorrelation_experiment,
    estimator_experiment,
    fz_check,
    paired_difference,
    projected_crps_experiment,
    projection_consistency_table,
    rearrangement_experiment,
    summarize_estimators,
    tail_experiment,
    transfer_experiment,
)
from sctrange.data import bulk_density_sct, discover_cases, load_case, read_prediction
from sctrange.download import ensure_dataset, usable_records
from sctrange.runs import run_complete
from sctrange.empirical import case_identities, summarize_identities
from sctrange.fields import gaussian_random_field
from sctrange.hlut import HLUT
from sctrange.plotting import (
    apply_style,
    plot_angle_energy,
    plot_autocorrelation_law,
    plot_estimator_bias,
    plot_spectral_weights,
    plot_tail_ecdf,
)
from sctrange.spectral import segment_energy_spectral, sliding_segment_sums
from sctrange.split import load_split, select_cases

In [ ]:
import os

from sctrange.runtime import frozen_split_path, resolve_paths

PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
S2_CASES = int(os.environ.get("SCT_MAX_CASES", S2_CASES)) or None
OUTPUT_DIR = PATHS.output_root / "s2"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20261013
CORRELATION_LENGTHS = (0, 1, 2, 4, 8, 16, 32, 64)
TAIL_CORRELATION_LENGTHS = (0, 4, 16)
SEGMENT_LENGTHS = (10, 60)
ANGLE_FREQUENCIES = np.geomspace(0.02, 0.15, 8)
LABELS = {"range, L = 10 mm": (0.447, 10), "range, L = 60 mm": (0.155, 10), "gradient loss": (0.2, -14), "voxel L2": (0.12, 10)}

hlut = HLUT.pyradplan_default()
checks: list[tuple[str, str, str, bool]] = []


def rng(section: int) -> np.random.Generator:
    return np.random.default_rng([SEED, section])


def record(section: str, claim: str, evidence: str, passed: bool) -> None:
    checks.append((section, claim, evidence, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {claim}: {evidence}")


apply_style()
pd.set_option("display.width", 160)

### 1 · Coherence, not magnitude, sets the range error

**Rearrangements.** Seven continuous errors of both signs are placed on seven voxels, three of which lie on the ray.
All 5040 arrangements are enumerated. Propositions 1 and 1′ predict that the extremes equal the sorted (Hardy–Littlewood)
bounds and that only arrangements ordering the errors like the weights reach the maximum (Bellini et al., 2022,
Lemma A.2).

In [ ]:
rearrangement = rearrangement_experiment(rng(12))
bound_gap = max((rearrangement["exhaustive_max"] - rearrangement["bound_max"]).abs().max(),
                (rearrangement["exhaustive_min"] - rearrangement["bound_min"]).abs().max())
record("1", "Propositions 1, 1′: exhaustive extremes equal the sorted bounds", f"max gap {bound_gap:.1e}", bound_gap < 1e-12)
record("1", "Only comonotone arrangements attain the maximum",
       f"{rearrangement['all_maximisers_comonotone'].sum()}/{len(rearrangement)} trials, "
       f"min gap to next value {rearrangement['gap_to_second_value'].min():.3f}",
       rearrangement["all_maximisers_comonotone"].all() and (rearrangement["gap_to_second_value"] > 0).all())
display(rearrangement.head().round(4))

**Autocorrelation law.** SPR errors are stationary Gaussian fields with standard deviation 0.03 (≈ 30 HU in soft tissue) and an exponential
correlation of length λ along and across the beam. The marginal law, hence the MAE, is identical for every λ.
Proposition 2 predicts the RMS error over a 100 mm segment from the exact circular covariance. The control shuffles
the same error values across all voxels and fields: identical MAE, white layout.

In [ ]:
coherence = autocorrelation_experiment(CORRELATION_LENGTHS, rng(1))
biased = autocorrelation_experiment((0, 8), rng(11), mean=0.002)

relative_error = (coherence["rms_observed_mm"] / coherence["rms_predicted_mm"] - 1).abs()
record("1", "Proposition 2 predicts RMS ΔWET", f"max relative error {relative_error.max():.1%}", relative_error.max() < 0.08)
record("1", "Bias term L²μ² adds as predicted",
       f"max relative error {(biased['rms_observed_mm'] / biased['rms_predicted_mm'] - 1).abs().max():.1%}",
       (biased["rms_observed_mm"] / biased["rms_predicted_mm"] - 1).abs().max() < 0.08)
mae_spread = coherence["mae_spr"].std() / coherence["mae_spr"].mean()
record("1", "MAE unchanged across correlation lengths", f"coefficient of variation {mae_spread:.1%}", mae_spread < 0.05)
gain = coherence["rms_observed_mm"].max() / coherence["rms_observed_mm"].min()
record("1", "Corollary 2.1: same MAE, several-fold range error", f"RMS ratio {gain:.1f}×", gain > 5)
shuffled = (coherence["rms_permuted_mm"] / coherence["rms_predicted_mm"].iloc[0] - 1).abs().max()
record("1", "Shuffled errors behave as white noise", f"max deviation {shuffled:.1%}", shuffled < 0.05)

display(coherence.round(4))
plot_autocorrelation_law(coherence).savefig(FIGURE_DIR / "fig1_coherence.png", bbox_inches="tight")
plt.show()

### 2 · Spectral weighting

**Segments.** Plane-wave errors along the beam are summed over circular segments of length L. Proposition 3 states an
exact identity: the measured transfer must equal the Dirichlet kernel to machine precision, and errors that vary only
across the beam must pass unattenuated. Voxel-wise L2 and a forward-difference gradient loss are shown for comparison.

In [ ]:
fields = gaussian_random_field((128, 48), 8.0, 4, rng(2))
parseval = max(
    abs(np.sum(sliding_segment_sums(field, length, axis=0) ** 2) - segment_energy_spectral(field, length, axis=0))
    / np.sum(sliding_segment_sums(field, length, axis=0) ** 2)
    for field in fields for length in (*SEGMENT_LENGTHS, 30)
)
record("2", "Proposition 3: Parseval identity on random fields", f"max relative error {parseval:.1e}", parseval < 1e-10)

transfer = transfer_experiment(SEGMENT_LENGTHS)
transfer_gap = float(np.max(np.abs(transfer["measured"] - transfer["theory"])))
record("2", "Measured transfer equals theory", f"max absolute gap {transfer_gap:.1e}", transfer_gap < 1e-10)
across = transfer[transfer["functional"] == "range, across beam"]["measured"]
record("2", "Errors varying across the beam pass unattenuated", f"transfer in [{across.min():.6f}, {across.max():.6f}]",
       np.allclose(across, 1.0))

landmarks = transfer[transfer["frequency"].isin([2 / 512, 8 / 512, 32 / 512, 128 / 512, 256 / 512])]
display(landmarks.pivot(index="frequency", columns="functional", values="theory").round(5))
plot_spectral_weights(transfer, LABELS).savefig(FIGURE_DIR / "fig2a_spectral_weights.png", bbox_inches="tight")
plt.show()

**All angles.** Windowed plane waves are projected with the differentiable projector over 180 angles. Proposition 3.1
predicts that the angle-summed range energy, relative to the voxel energy, decays as 1/|k|: a log-log slope of −1.

In [ ]:
angles, slope = angle_energy_experiment(ANGLE_FREQUENCIES, rng(3))
record("2", "Proposition 3.1: angle-summed energy ∝ 1/|k|", f"log-log slope {slope:.3f}", abs(slope + 1) < 0.15)
display(angles.round(3))
plot_angle_energy(angles, slope).savefig(FIGURE_DIR / "fig2b_angle_energy.png", bbox_inches="tight")
plt.show()

### 3 · Bayes acts under range losses

Each voxel is soft tissue, or with probability pᵢ an alternative tissue (gas, or bone in the second scenario); pᵢ is a
smooth map of blobs peaking at 0.45, and a copula of correlation length 8 mm couples the classes. Three point estimates are
computed from the exact marginals: the HU mean (Bayes act of L2 in HU), the HU median (Bayes act of L1, in HU or SPR) and
the SPR mean g⁻¹(E[g(Y)]), the Bayes act of the squared range loss (Proposition 4). Segments run from depth 0 to 40, 80 and
120 mm, one per column.

For every ray the Monte Carlo risk over 3000 posterior draws is compared with the analytic decomposition
risk = bias² + variance, where the bias is computed from the exact marginals alone.

In [ ]:
estimator_detail = pd.concat([estimator_experiment(hlut, scenario, rng(4 + i)) for i, scenario in enumerate(("gas", "bone"))])
estimator_summary = summarize_estimators(estimator_detail)

bayes = estimator_summary.xs("SPR mean (Bayes act)", level="estimator")
record("3", "Bayes act has zero range bias", f"max |mean bias| {bayes['mean_bias_mm'].abs().max():.1e} mm",
       bayes["mean_bias_mm"].abs().max() < 1e-9)
lowest = estimator_summary.groupby(["scenario", "depth_mm"])["rms_mc_mm"].idxmin().map(lambda index: index[1])
record("3", "Bayes act has the lowest Monte Carlo range risk", f"{(lowest == 'SPR mean (Bayes act)').sum()}/{len(lowest)} settings",
       (lowest == "SPR mean (Bayes act)").all())
decomposition = (estimator_summary["rms_mc_mm"] / estimator_summary["rms_predicted_mm"] - 1).abs().max()
record("3", "Risk = analytic bias² + variance", f"max relative gap {decomposition:.1%}", decomposition < 0.02)
median_bias = estimator_summary.xs("HU median (L1)", level="estimator")["mean_bias_mm"]
record("3", "Proposition 5: L1 over-estimates SPR next to uncertain gas",
       f"mean bias {median_bias.loc['gas'].min():.1f} to {median_bias.loc['gas'].max():.1f} mm", (median_bias.loc["gas"] > 0).all())
record("3", "Proposition 5: L1 under-estimates SPR next to uncertain bone",
       f"mean bias {median_bias.loc['bone'].min():.1f} to {median_bias.loc['bone'].max():.1f} mm", (median_bias.loc["bone"] < 0).all())

display(estimator_summary.round(3))
plot_estimator_bias(estimator_detail[estimator_detail["depth_mm"] == 120]).savefig(
    FIGURE_DIR / "fig3a_estimator_bias.png", bbox_inches="tight")
plt.show()

### 4 · Only the mean is consistent under projection

Six copies of the uniform law on a small HU multiset are coupled either cyclically (the joint mix of Theorem 6, whose sum
is constant) or independently (the law of the sum is enumerated exactly over 6⁶ outcomes). A functional is
projection-consistent when its value on the sum equals the sum of its voxel values under every coupling.

In [ ]:
consistency = projection_consistency_table([-850.0, 40.0, 40.0, 40.0, 40.0, 700.0]).set_index("functional")
couplings = ["value on sum, cyclic coupling", "value on sum, independent coupling"]
matches = consistency[couplings].sub(consistency["sum of voxel values"], axis=0).abs().max(axis=1) < 1e-9
record("4", "Mean is additive under both couplings", f"max gap {consistency.loc['mean', couplings].sub(consistency.loc['mean', 'sum of voxel values']).abs().max():.1e}",
       matches["mean"])
record("4", "Median, quantile and expectiles are not", f"{(~matches.drop('mean')).sum()}/{len(matches) - 1} inconsistent",
       (~matches.drop("mean")).all())
display(consistency.round(2))

### 5 · Tails of range error need the joint law

The gas posterior is sampled with three copula correlation lengths. Voxel marginals, and therefore the Bayes act, are
identical in all three; only the dependence between voxels changes. The full-depth range error of the same point estimate
is then compared. The FZ0 score, minimised over a grid that does not contain the empirical values, must recover the
empirical VaR and ES of |ΔWET| within one grid step.

In [ ]:
tails, tail_errors = tail_experiment(hlut, TAIL_CORRELATION_LENGTHS, rng(6))
fraction_gap = tails["alternative_fraction"].max() - tails["alternative_fraction"].min()
record("5", "Identical marginals across copulas", f"gas fraction range {fraction_gap:.3f}", fraction_gap < 0.01)
var_ratio = tails["VaR90_mm"].iloc[-1] / tails["VaR90_mm"].iloc[0]
record("5", "Proposition 7: same point estimate, different tail", f"VaR90 ratio {var_ratio:.1f}×", var_ratio > 2)

fz = fz_check(tail_errors[TAIL_CORRELATION_LENGTHS[-1]], rng=rng(7))
fz_ok = (abs(fz["FZ argmin VaR (mm)"] - fz["empirical VaR (mm)"]) <= 1.01 * fz["grid step VaR (mm)"]
         and abs(fz["FZ argmin ES (mm)"] - fz["empirical ES (mm)"]) <= 1.01 * fz["grid step ES (mm)"])
record("5", "FZ0 score elicits (VaR, ES) of |ΔWET|", "within one grid step", fz_ok)

display(tails.round(3))
display(pd.Series(fz, name="value").to_frame().round(3))
plot_tail_ecdf(tail_errors).savefig(FIGURE_DIR / "fig3b_tails.png", bbox_inches="tight")
plt.show()

### 6 · Evaluating generative predictions

Observations are drawn from the gas posterior with copula length 16 mm. Two ensemble forecasts with the exact voxel
marginals are scored: the correct model, and a model with independent voxels. Voxel-wise CRPS averages one-dimensional
scores per voxel; projected CRPS scores the law of each full-depth range integral. Differences are paired over
observations, with percentile bootstrap intervals.

In [ ]:
crps = projected_crps_experiment(hlut, rng(8))
voxel = paired_difference(crps, "voxel CRPS, correct", "voxel CRPS, wrong dependence", rng(9))
projected = paired_difference(crps, "projected CRPS, correct", "projected CRPS, wrong dependence", rng(10))
record("6", "Voxel CRPS cannot tell the two forecasts apart",
       f"95% CI [{voxel['ci_low']:.4f}, {voxel['ci_high']:.4f}] contains 0", voxel["ci_low"] < 0 < voxel["ci_high"])
record("6", "Projected CRPS penalises the wrong dependence",
       f"95% CI [{projected['ci_low']:.2f}, {projected['ci_high']:.2f}] mm, {projected['relative to score']:.0%} worse",
       projected["ci_low"] > 0)
display(pd.DataFrame({"voxel CRPS (SPR)": voxel, "projected CRPS (mm)": projected}).T.round(4))

In [ ]:
def write_report(real_data: dict | None) -> pd.DataFrame:
    """``s2_report.json`` with every verdict so far: written now (sections 1–6) and again after section 7, so the
    synthetic checks are kept even if the real-data section cannot run."""
    verdicts = pd.DataFrame(checks, columns=["section", "claim", "evidence", "passed"])
    report = {"seed": SEED, "hlut_fingerprint": hlut.fingerprint(), "real_data": real_data,
              "complete": real_data is not None, "checks": verdicts.to_dict(orient="records"),
              "all_passed": bool(verdicts["passed"].all())}
    (OUTPUT_DIR / "s2_report.json").write_text(json.dumps(report, indent=2))
    estimator_summary.to_csv(OUTPUT_DIR / "estimator_summary.csv")
    return verdicts


write_report(None)
print(f"sections 1–6 saved to {OUTPUT_DIR / 's2_report.json'}; section 7 completes the report")


### 7 · The identities on real patients

Propositions 1, 1′, 2 and 3 are identities: they hold for every error field, so they can be checked exactly on real
data. For each development case of `TASK` (frozen split of S1; test cases are never loaded), the SPR error of every
available synthetic CT — the bulk-density baseline of the challenge and any out-of-fold prediction of S3 or S4 — is
taken on 16 slices around the largest body section, with beams along the two in-plane axes (0° and 90°) and circular
segments of 10 and 60 mm.

- **Proposition 3**: the energy of segment sums equals the Dirichlet-weighted spectrum of the real error;
- **Proposition 2**: with the mean μ, variance σ² and autocorrelation measured on the same error, L²μ² + Lσ²l is the
  mean squared segment error; the effective length *l* (1 sample for white errors) is the coherence of real errors;
- **Propositions 1, 1′**: the range error of every ray of the largest slice lies between the sums of the *n* smallest
  and *n* largest voxel errors of that slice; its *position* between them (0 = lower, 1 = upper bound) shows how close
  the real layout comes to the worst case.

`coherence gain` = RMS segment error / RMS of white errors with the same μ and σ: the factor by which the spatial
layout of real errors, not their magnitude, inflates range error (Corollary 2.1).

In [ ]:
DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(SPLIT_PATH)
development, excluded = usable_records(DATA_ROOT, [case for case in discover_cases(DATA_ROOT)
                                                  if case.task == TASK and case.case_id in set(split["development"])])
if S2_CASES:  # a region-stratified subset, not the first case ids (which all come from one region and centre)
    keep = set(select_cases(development, S2_CASES, SEED))
    development = [case for case in development if case.case_id in keep]
if not development:
    raise RuntimeError(f"no development case of task {TASK} under {DATA_ROOT}")
ids = [case.case_id for case in development]
PREDICTIONS = {f"{stage} {run.name}": run / "predictions"  # finished runs only
               for stage, root in (("S3", PATHS.output_root / "s3" / "runs"), ("S4", PATHS.output_root / "s4" / "runs" / "seed0"))
               for run in sorted(root.glob("*")) if run.is_dir() and run_complete(run, ids)}

real_rows = []
for index, case in enumerate(development, 1):
    volumes = load_case(case, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sources = {"bulk density": bulk_density_sct(mask)}
    for name, directory in PREDICTIONS.items():
        if (directory / f"{case.case_id}.mha").exists():
            sources[name] = read_prediction(directory / f"{case.case_id}.mha", ct.shape)
    for source, sct in sources.items():
        for row in case_identities(ct, mask, sct, spacing[0], hlut):
            real_rows.append({"case_id": case.case_id, "region": case.region, "source": source, **row})
    if index % 10 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

real = pd.DataFrame(real_rows)
real.to_csv(OUTPUT_DIR / "real_identities.csv", index=False)
real_summary = {"task": TASK, "sources": sorted(real["source"].unique()), **summarize_identities(real)}
record("7", "Proposition 3 on patients: Parseval identity", f"max relative gap {real_summary['max_parseval_gap']:.1e}",
       real_summary["max_parseval_gap"] < 1e-8)
record("7", "Proposition 2 on patients: L²μ² + Lσ²l with the measured moments",
       f"max relative gap {real_summary['max_decomposition_gap']:.1e}", real_summary["max_decomposition_gap"] < 1e-8)
record("7", "Propositions 1, 1′ on patients: every ray within the sorted bounds",
       f"{real_summary['violations']} violations over {real_summary['rays']} rays", real_summary["violations"] == 0)

segments = real[real["check"] == "segments"]
display(segments.groupby(["source", "segment_mm", "beam"])[["mean_spr", "std_spr", "effective_length_mm", "rms_mm",
                                                            "white_rms_mm"]].median()
        .assign(coherence_gain=lambda f: f["rms_mm"] / f["white_rms_mm"]).round(4))
rays = real[real["check"] == "rays"]
display(rays.groupby(["source", "beam"])[["median_position", "max_position", "max_abs_mm", "worst_case_mm"]]
        .median().round(3))
print(f"{real_summary['cases']} development cases of task {TASK} · sources: {', '.join(real_summary['sources'])}")

### 8 · Verdicts

In [ ]:
verdicts = write_report(real_summary)
display(verdicts)
print(f"{verdicts['passed'].sum()}/{len(verdicts)} checks passed")

### Consequences for S3–S5

Sections 1–6 hold on the synthetic benchmark, where the posterior is known; section 7 confirms the identities on
real patients and measures the coherence of real errors. The conclusions below are hypotheses that S3 and S4 test
on real data.

1. **Regression target.** The squared-error regressor in SPR space is the Bayes act for mean squared range error. It
   becomes arm R3 and the reference every other arm must beat. An L1 loss, the default in sCT work, carries a
   signed range bias wherever the posterior is asymmetric, with a sign set by the tissue pair.
2. **Role of the range loss.** At the Bayes act no loss can reduce mean squared range error further, so the projected
   loss can only help through finite capacity (reallocating residual errors away from coherent, low-frequency patterns)
   and through the tails. Its claimed benefit must be measured on CVaR at matched MAE, against R3 and the per-tissue bias
   correction R4.
3. **Tails.** Range-error tails depend on dependence between voxels that no point estimate encodes. They require a
   joint model (the stochastic-interpolant flow) or a direct VaR/ES head trained with the FZ score.
4. **Evaluation.** Generative sCT models must be compared with projected CRPS; voxel-wise CRPS is blind to exactly the
   property that governs range.
5. **Diagnostic for S3.** On real sCTs, the shuffling controls and the spectral split of the error decide which
   mechanism dominates: coherent tissue-level biases (range loss and bias correction matter) or interface blur
   (largely invisible to range).

In [ ]:
PIPELINE.finish("S2", complete=bool(True), note=str(""))


# S3 · Range diagnostics on real sCTs (go/no-go 1)

Week 3 asks one question, fixed in advance in `configs/s3_preregistration.json`:

> Do out-of-fold sCT errors on SynthRAD2025 development cases carry excess low-frequency energy **along the beam**,
> relative to the same errors shuffled?

If they do, the range-aware loss of weeks 4–5 has something to act on (docs/theory.md, Result A and Proposition 2).
If they do not, range error is dominated by white or interface errors that voxel losses already see, and the paper
re-centres on Results B–C (projection consistency, tails, proper evaluation), which do not depend on this outcome.

Pipeline:

1. verify the frozen split and the pre-registration;
2. train 2.5D U-Nets with two point losses, **cross-fitted** on the development pool so that every diagnosed prediction
   is out of fold; test cases are never loaded;
3. per case and source: range metrics with permutation controls, and tapered periodograms of the SPR error along and
   across beams, relative to shuffled errors;
4. apply the pre-registered rule, then the secondary tests.

Sources: the bulk-density baseline, `l1_hu` (L1 in HU, the usual sCT loss, Bayes act = median) and `l2_spr` (L2 in SPR,
Bayes act for squared range error, Proposition 4).

Everything runs from this notebook, locally or on Colab: download, cache, cross-fitted training, diagnostics and the
verdict. Without a GPU, training is skipped and existing predictions are diagnosed; with none, only the bulk-density
baseline is diagnosed and the verdict is NOT EVALUATED.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead). `SCT_TASKS` selects the tasks (default `1,2`).

The notebook runs the code of this repository (`src/`), whether or not the package is installed.

*Stage notebook: `notebooks/s3_real_diagnostic.ipynb`.*


In [ ]:
PIPELINE.begin("S3", {'task': TASK, 'steps': STEPS, 'folds': FOLDS, 'train': TRAIN}, PIPELINE_OUTPUTS / "s3" / "s3_report.json", after=('S1',))


In [ ]:
import json
import os
import warnings
from dataclasses import asdict, replace
from pathlib import Path

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.data import bulk_density_sct, discover_cases, load_case, read_prediction
from sctrange.diagnostics import Preregistration, decide, diagnose_case, paired_test
from sctrange.evaluation import DiagnosticConfig
from sctrange.hlut import HLUT
from sctrange.plotting import apply_style, plot_beam_spectra, plot_signed_bias, plot_training_curves
from sctrange.download import ensure_dataset, usable_records
from sctrange.runs import cached_result, config_key, files_signature, run_complete
from sctrange.training import visible_devices
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_split, split_hash
from sctrange.training import TrainConfig, crossfit

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
TRAIN = os.environ.get("SCT_TRAIN_IN_NOTEBOOK", "1" if TRAIN else "0") == "1"
STEPS = int(os.environ.get("SCT_STEPS", STEPS))
N_FOLDS = int(os.environ.get("SCT_FOLDS", FOLDS))

OUTPUT_DIR = PATHS.output_root / "s3"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_RUNS_ROOT", OUTPUT_DIR / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", PATHS.cache_root / "cache"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s3_preregistration.json"

SEED = 20261013
DEVICES = visible_devices()  # ["cuda:0", "cuda:1"] on Kaggle T4 x 2
DEVICE = DEVICES[0]
LOSSES = ("l1_hu", "l2_spr")
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
MAX_CASES = None
hlut = HLUT.pyradplan_default()

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
TRAIN_IN_NOTEBOOK = TRAIN and (DEVICE.startswith("cuda") or os.environ.get("SCT_TRAIN_ON_CPU") == "1")
if TRAIN and not TRAIN_IN_NOTEBOOK:
    print("no GPU: training skipped, existing predictions only (Colab: Runtime > Change runtime type > GPU; "
          "SCT_TRAIN_ON_CPU=1 forces CPU training)")
SPLIT_PATH = FROZEN_SPLIT_PATH
TRAIN_CONFIG = TrainConfig(seed=SEED, device=DEVICE, steps=STEPS)

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
print(f"task {TASK} · devices {', '.join(DEVICES)} · folds {N_FOLDS} · steps {STEPS} · train in notebook: {TRAIN_IN_NOTEBOOK}")

### 1 · Frozen split and pre-registration

The split must already exist on real data (week 1) and is only verified here. The pre-registration file is hashed and
the hash goes into the report, so the decision rule demonstrably predates the results.

In [ ]:
records = discover_cases(DATA_ROOT)
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(SPLIT_PATH)

development_ids = set(split["development"])
development, excluded = usable_records(DATA_ROOT, [r for r in records
                                                  if r.case_id in development_ids and r.task == TASK])
development = development[:MAX_CASES]
for case_id, problems in excluded.items():
    print(f"excluded {case_id}: {problems}")
assert not development_ids & set(split["test"])

prereg = Preregistration.load(PREREGISTRATION_PATH)
display(pd.Series({"split sha256": split_hash(split), "development cases (this task)": len(development), "excluded (integrity check)": len(excluded),
                   "test cases (never loaded)": len(split["test"]),
                   "pre-registration sha256": prereg.fingerprint}, name="value").to_frame())
display(pd.json_normalize(prereg.raw, sep=" · ").T.rename(columns={0: "value"}))

**Rule.** For the primary source (`l1_hu`), over development cases:

| Verdict | Condition |
|---|---|
| GO | median along-beam low-frequency excess ≥ 1.5 **and** median target-segment global-shuffle ratio ≥ 1.2 |
| WEAK | median along-beam low-frequency excess ≥ 1.15 |
| NO-GO | otherwise |

*Low-frequency excess* is the share of error power below 0.05 cycles/mm (periods above 20 mm, organ scale) in 64 mm
windows along the beam (beams at 0°, 60° and 120°), divided by the same share for the errors shuffled over the body;
1 means white-like layout.
The *global-shuffle ratio* is the RMS range error to the distal target edge divided by that of the shuffled errors at
identical MAE. A uniform bias leaves both statistics at 1 by construction (the shuffle preserves the mean); it is
reported separately as the signed range error.

### 2 · Cross-fitted training

Development cases are cached once (normalised, body-cropped, memory-mapped), then split into stratified folds. For each
loss and fold, a model is trained on the other folds and predicts the held-out fold. Interrupted runs resume from the
saved fold checkpoints, and a fold cut mid-training from its last in-fold save (every 1000 steps). A run counts
as available only once all its out-of-fold predictions are written. On real data, train with the CLI on a GPU and rerun this notebook:

```bash
for loss in l1_hu l2_spr; do
  sctrange-train --data-root "$SYNTHRAD2025_ROOT" --split splits/synthrad2025_local_test.json \
    --cache-dir outputs/cache --out outputs/s3/runs --loss $loss --task 1 --folds 5
done
```

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")

histories = {}
ids = [record.case_id for record in development]
for loss in LOSSES:
    run_dir = RUNS_ROOT / loss
    if TRAIN_IN_NOTEBOOK:
        crossfit(development, cache, replace(TRAIN_CONFIG, loss=loss), hlut, N_FOLDS, run_dir, DEVICES)
    elif not run_complete(run_dir, ids):
        print(f"{loss}: run not finished under {run_dir}; run this notebook on a GPU to train it")
        continue
    if (run_dir / "history.csv").exists():
        histories[loss] = pd.read_csv(run_dir / "history.csv")

COMPLETE_RUNS = [loss for loss in LOSSES if run_complete(RUNS_ROOT / loss, ids)]
display(pd.Series(asdict(TRAIN_CONFIG), name="value").to_frame())
if histories:
    plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()
    display(pd.concat({loss: h.groupby("fold").tail(1).set_index("fold")[["step", "loss", "seconds"]]
                       for loss, h in histories.items()}).round(3))

### 3 · Per-case diagnostics

For every development case: the bulk-density baseline and every available out-of-fold prediction are compared with the
reference CT. Range metrics use 8 coplanar beams and a 25 mm surrogate target at the body centroid (SynthRAD2025
releases no PTV). Spectra use 64 mm windows lying entirely inside the body, on three beams 60° apart (0°, 60°, 120°):
with beams every 45°, the across direction of one beam is the along direction of another and the two spectra would be
identical by construction.

In [ ]:


def diagnose(record, path):
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sct = bulk_density_sct(mask) if path is None else read_prediction(path, ct.shape)
    return diagnose_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC, prereg)


rows, spectra = [], {}
for index, record in enumerate(development, 1):  # each result cached: an interrupted session resumes here
    paths = {"bulk density": None,
             **{loss: RUNS_ROOT / loss / "predictions" / f"{record.case_id}.mha" for loss in COMPLETE_RUNS}}
    data = files_signature([record.path("ct"), record.path("mask")], mtime=False)
    for source, path in paths.items():
        key = {"config": config_key(DIAGNOSTIC), "prereg": prereg.fingerprint, "hlut": hlut.fingerprint(),
               "data": data, "prediction": files_signature([path]) if path else None}
        metrics, case_spectra = cached_result(
            OUTPUT_DIR / "cache_eval" / f"{record.case_id}_{source.replace(' ', '_')}.pkl", key,
            lambda: diagnose(record, path))
        rows.append({"case_id": record.case_id, "region": record.region, "center": record.center,
                     "source": source, **metrics})
        spectra.setdefault(source, []).append(case_spectra)
    if index % 50 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

diagnostics = pd.DataFrame(rows)
diagnostics.to_csv(OUTPUT_DIR / "s3_diagnostics.csv", index=False)
print(f"{diagnostics['case_id'].nunique()} cases, {len(diagnostics)} evaluations, sources: {sorted(spectra)}")

columns = ["mae_hu", "target_mean_signed_mm", "target_rms_mm", "target_cvar_mm", "target_coherence",
           "target_intra_ratio", "target_global_ratio", "lf_excess_along", "lf_excess_across"]
display(diagnostics.groupby(["source", "region"])[columns].median().round(3))

Mean error power, as a share of total power, relative to the same share for shuffled errors. Values above 1 at low
frequency (grey band) mean coherent, organ-scale errors; a curve that rises along the beam but not across it would point
to beam-direction structure, a flat one to isotropic coherence.

In [ ]:
plot_beam_spectra(spectra, prereg.cutoff).savefig(FIGURE_DIR / "beam_spectra.png", bbox_inches="tight")
plt.show()
plot_signed_bias(diagnostics).savefig(FIGURE_DIR / "signed_bias.png", bbox_inches="tight")
plt.show()

### 4 · Pre-registered decision and secondary tests

Secondary tests are reported whatever the verdict and do not change it:

- **Proposition 5 on real data**: in the abdomen (uncertain gas), the signed range error of `l1_hu` exceeds that of
  `l2_spr` (paired one-sided Wilcoxon over cases);
- **tails**: target CVaR90 of `l1_hu` versus `l2_spr` (two-sided);
- **image quality**: MAE of `l1_hu` versus `l2_spr` (two-sided), to read the two previous results at comparable MAE.

In [ ]:
decision = decide(diagnostics, prereg)
secondary = prereg.raw["secondary"]
first, second = secondary["bias_pair"]
tests = {
    "signed bias, AB (l1_hu − l2_spr > 0)": paired_test(diagnostics, secondary["bias_column"], first, second,
                                                       secondary["bias_region"], secondary["bias_alternative"]),
    "target CVaR90 (l1_hu − l2_spr)": paired_test(diagnostics, secondary["tail_column"], first, second),
    "MAE (l1_hu − l2_spr)": paired_test(diagnostics, "mae_hu", first, second),
}
display(pd.Series(decision, name="value").to_frame())
display(pd.DataFrame(tests).T.round(4))

### 5 · Report

In [ ]:
report = {
    "task": TASK,
    "split_sha256": split_hash(split),
    "train_config": asdict(TRAIN_CONFIG),
    "folds": N_FOLDS,
    "decision": decision,
    "secondary": tests,
    "medians_by_source": diagnostics.groupby("source")[columns].median().to_dict(orient="index"),
}
(OUTPUT_DIR / "s3_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Pre-registration hashed", prereg.fingerprint[:12], True),
    ("Out-of-fold predictions", ", ".join(s for s in spectra if s != "bulk density") or "none",
     all(loss in spectra for loss in LOSSES)),
    ("Diagnostics", f"{len(diagnostics)} evaluations", len(diagnostics) > 0),
    ("Verdict on real data", decision["verdict"], decision["verdict"] != "NOT EVALUATED"),
], columns=["item", "detail", "passed"])
display(status)

### What each verdict implies for weeks 4–5

| Verdict | Weeks 4–5 |
|---|---|
| GO | Run arms R1–R6 as planned. The range loss (R5) and the VaR/ES head (R6) are tested against R3 (L2 in SPR) and R4 (R3 + per-tissue bias correction) on CVaR at matched MAE. |
| WEAK | Keep R1–R4 and R6; run R5 with a reduced sweep. Report the effect size of coherence rather than a headline gain. |
| NO-GO | Drop R5 as a contribution. The paper rests on Results B–C: estimator choice (R1 vs R3), tails (R6 and the generative arm), projected CRPS. |

In every case, the secondary bias test decides whether the L1-versus-SPR-mean result of Proposition 5 holds on patients,
which is reportable on its own.

Weeks 4–5 run in `notebooks/s4_regression_arms.ipynb`, with their own pre-registered rule (`configs/s4_preregistration.json`).

In [ ]:
PIPELINE.finish("S3", complete=bool(decision["verdict"] != "NOT EVALUATED" and all(loss in spectra for loss in LOSSES)), note=str(decision["verdict"]))


# S4–S5 · Regression arms at a strictly matched budget

Weeks 4–5 compare six regression arms that share the backbone (2.5D U-Net), the data, the folds, the number of steps and
the seeds; only the objective changes.

| Arm | Objective | Role |
|---|---|---|
| R1 | L1 in HU | current practice (Bayes act: median) |
| R2 | L2 in HU | control (mean in HU) |
| R3 | L2 in SPR | Bayes act for squared range error (docs/theory.md, Proposition 4) |
| R4 | R3 + per-tissue SPR offset, cross-fitted | mandatory cheap control |
| R5 | R3 + λ · mean ΔWET² over plan beams and random directions | method |
| R6 | R5 + β · FZ0 score of a per-ray (VaR, ES) head | method, tails |

The question and the decision rule are fixed in `configs/s4_preregistration.json` before real predictions exist:
does R5 or R6 lower the CVaR90 of the range error to the distal target edge relative to R3 at an MAE at most 5 % above R3's, and
does it beat the bias-corrected R4? If only R3 is beaten, the gain is a tissue-level bias and *the right estimator
suffices*, which is publishable on its own (plan, S6). The verdict uses the range proxy; the γ-proton part of go/no-go 2
needs the S6 dosimetry subset.

Everything runs from this notebook, locally or on Colab: download, cache, training of R1–R3, R5 and R6 for every
seed (runs under `outputs/s4/runs/seed<k>/<arm>`), the R4 correction, evaluation and the verdict. The full budget is
5 arms × 5 folds × 3 seeds; every fold is checkpointed, so it can be spread over several Colab sessions. Without a GPU,
training is skipped and existing runs are evaluated; with none, the notebook stops with a clear message.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead). `SCT_TASKS` selects the tasks (default `1,2`).

*Stage notebook: `notebooks/s4_regression_arms.ipynb`.*


In [ ]:
PIPELINE.begin("S4", {'task': TASK, 'steps': STEPS, 'folds': FOLDS, 'seeds': SEED_LIST, 'train': TRAIN}, PIPELINE_OUTPUTS / "s4" / "s4_report.json", after=('S1',))


In [ ]:
import json
import os
import warnings
from dataclasses import asdict

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.correction import crossfit_bias_correction
from sctrange.data import discover_cases, load_case, read_prediction
from sctrange.diagnostics import Preregistration, decide_arms, holm, paired_test
from sctrange.evaluation import DiagnosticConfig, evaluate_case, evaluate_risk
from sctrange.hlut import HLUT
from sctrange.plotting import apply_style, plot_arm_pareto, plot_training_curves
from sctrange.download import ensure_dataset, usable_records
from sctrange.runs import cached_result, config_key, files_signature, run_complete
from sctrange.training import visible_devices
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_split, split_hash
from sctrange.training import ARMS, arm_config, crossfit_runs

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
TRAIN = os.environ.get("SCT_TRAIN_IN_NOTEBOOK", "1" if TRAIN else "0") == "1"
STEPS = int(os.environ.get("SCT_STEPS", STEPS))
N_FOLDS = int(os.environ.get("SCT_FOLDS", FOLDS))
SEEDS = tuple(int(s) for s in os.environ.get("SCT_SEEDS", SEED_LIST).split(","))

OUTPUT_DIR = PATHS.output_root / "s4"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_RUNS_ROOT", OUTPUT_DIR / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", PATHS.cache_root / "cache"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s4_preregistration.json"

SEED = 20261013
DEVICES = visible_devices()  # ["cuda:0", "cuda:1"] on Kaggle T4 x 2
DEVICE = DEVICES[0]
TRAINED_ARMS = tuple(sorted(ARMS))
ARM_ORDER = ("R1", "R2", "R3", "R4", "R5", "R6")
PREREGISTERED_SEEDS = (0, 1, 2)  # the verdict is preliminary on fewer seeds
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
MAX_CASES = None
OVERRIDES: dict = {"steps": STEPS}
hlut = HLUT.pyradplan_default()

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
TRAIN_IN_NOTEBOOK = TRAIN and (DEVICE.startswith("cuda") or os.environ.get("SCT_TRAIN_ON_CPU") == "1")
if TRAIN and not TRAIN_IN_NOTEBOOK:
    print("no GPU: training skipped, existing runs only (Colab: Runtime > Change runtime type > GPU; "
          "SCT_TRAIN_ON_CPU=1 forces CPU training)")
SPLIT_PATH = FROZEN_SPLIT_PATH

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
print(f"task {TASK} · devices {', '.join(DEVICES)} · folds {N_FOLDS} · steps {STEPS} · seeds {SEEDS} · "
      f"train in notebook: {TRAIN_IN_NOTEBOOK}")

### 1 · Frozen split and pre-registration

The frozen split of week 1 is verified, never regenerated on real data; test cases are never loaded. The
pre-registration file is hashed into the report.

In [ ]:
records = discover_cases(DATA_ROOT)
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(SPLIT_PATH)

development_ids = set(split["development"])
development, excluded = usable_records(DATA_ROOT, [r for r in records
                                                  if r.case_id in development_ids and r.task == TASK])
development = development[:MAX_CASES]
for case_id, problems in excluded.items():
    print(f"excluded {case_id}: {problems}")
assert not development_ids & set(split["test"])

prereg = Preregistration.load(PREREGISTRATION_PATH)
display(pd.Series({"split sha256": split_hash(split), "development cases (this task)": len(development), "excluded (integrity check)": len(excluded),
                   "test cases (never loaded)": len(split["test"]),
                   "pre-registration sha256": prereg.fingerprint}, name="value").to_frame())
display(pd.json_normalize({k: v for k, v in prereg.raw.items() if k != "verdicts"}, sep=" · ").T.rename(columns={0: "value"}))

### 2 · Training (cross-fitted) and the R4 control

For every seed, each trained arm is cross-fitted on the development pool; R4 is then derived from the out-of-fold R3
predictions, with offsets for fold *k* estimated on the other folds only. Checkpoints are reused when they match the
configuration and the training cases and refused otherwise, so reruns are cheap and arms cannot be mixed up; a fold cut
mid-training resumes from its last in-fold save. Seeds run one after the other, and within a seed the folds of
the five trained arms share one queue, so with two GPUs neither waits for the last fold of a run. A run that is
already complete is not predicted again, so its cached evaluations and its R4 correction stay valid. An arm counts as present only once all its predictions are written.

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")

histories, missing, offsets = {}, [], {}
ids = [record.case_id for record in development]
for seed in SEEDS:  # seed by seed: every arm of the first seed is complete before the next seed starts
    seed_dir = RUNS_ROOT / f"seed{seed}"
    if TRAIN_IN_NOTEBOOK:  # the folds of the five trained arms share one queue: a free GPU takes the next fold
        crossfit_runs(development, cache, [(arm_config(arm, seed=seed, device=DEVICE, **OVERRIDES), seed_dir / arm)
                                           for arm in TRAINED_ARMS], hlut, N_FOLDS, DEVICES)
    for arm in TRAINED_ARMS:
        run_dir = seed_dir / arm
        if not run_complete(run_dir, ids):
            missing.append(f"seed{seed}/{arm}")
        elif (run_dir / "history.csv").exists() and seed == SEEDS[0]:
            histories[arm] = pd.read_csv(run_dir / "history.csv")
    if run_complete(seed_dir / "R3", ids):  # recomputed only when the R3 predictions changed
        offsets[seed] = crossfit_bias_correction(development, seed_dir / "R3", seed_dir / "R4", hlut)
    else:
        missing.append(f"seed{seed}/R4 (needs R3)")

print("missing runs:", ", ".join(missing) if missing else "none")
used = sum(f.stat().st_size for f in RUNS_ROOT.rglob("*") if f.is_file()) / 2**30 if RUNS_ROOT.exists() else 0.0
runs_total, runs_done = len(SEEDS) * (len(TRAINED_ARMS) + 1), len(SEEDS) * (len(TRAINED_ARMS) + 1) - len(missing)
projected = f", about {used / runs_done * runs_total:.1f} GiB once all {runs_total} runs exist" if runs_done else ""
print(f"S4 runs use {used:.1f} GiB{projected}; {shutil.disk_usage(RUNS_ROOT if RUNS_ROOT.exists() else PATHS.output_root).free / 2**30:.1f} GiB free there")
display(pd.Series(asdict(arm_config("R6", seed=SEEDS[0], device=DEVICE, **OVERRIDES)), name="R6 configuration").to_frame())
if offsets:
    display(pd.concat(offsets, names=["seed"]).round(4).rename_axis(columns="R4 SPR offset by predicted class"))
if histories:
    plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()

### 3 · Per-case evaluation

Every out-of-fold prediction of every arm and seed is evaluated with the S1 metrics (8 coplanar beams, 25 mm surrogate
target at the body centroid). R6 risk maps are integrated along the same segments to check the calibration of the
predicted VaR90 and ES90. Per-case metrics are averaged over seeds before any test, using only the seeds whose six arms are all
complete (averaging arms over different seeds would compare them at unequal budgets). With fewer seeds than the
pre-registered 0, 1, 2 the verdict is marked preliminary: run with `SEEDS = "0"` first to get one after the first
seed, then with `"0,1,2"` (the seed-0 runs are reused). Each evaluation is cached in `outputs/s4/cache_eval/`, so an interrupted session resumes where it stopped.

In [ ]:


def evaluate_source(record, path, risk_paths):
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sct = read_prediction(path, ct.shape)
    row = evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)
    if not risk_paths:
        return row, None
    var, excess = (read_prediction(p, ct.shape) for p in risk_paths)
    return row, evaluate_risk(ct, mask, sct, var, excess, spacing, hlut, DIAGNOSTIC)


finished = [(seed, arm) for seed in SEEDS for arm in ARM_ORDER if run_complete(RUNS_ROOT / f"seed{seed}" / arm, ids)]
rows, risk_rows = [], []
for index, record in enumerate(development, 1):  # each result cached: an interrupted session resumes here
    data = files_signature([record.path("ct"), record.path("mask")], mtime=False)
    for seed, arm in finished:
        run_dir = RUNS_ROOT / f"seed{seed}" / arm
        path = run_dir / "predictions" / f"{record.case_id}.mha"
        risk_paths = [run_dir / "risk" / f"{record.case_id}_{name}.mha" for name in ("var", "excess")]
        risk_paths = risk_paths if all(p.exists() for p in risk_paths) else []
        key = {"config": config_key(DIAGNOSTIC), "hlut": hlut.fingerprint(), "data": data,
               "inputs": files_signature([path, *risk_paths])}
        row, risk_row = cached_result(OUTPUT_DIR / "cache_eval" / f"seed{seed}" / arm / f"{record.case_id}.pkl", key,
                                      lambda: evaluate_source(record, path, risk_paths))
        keys = {"case_id": record.case_id, "region": record.region, "center": record.center, "seed": seed, "source": arm}
        rows.append({**keys, **row})
        if risk_row is not None:
            risk_rows.append({**keys, **risk_row})
    if index % 50 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

if not rows:
    raise RuntimeError(f"no out-of-fold prediction under {RUNS_ROOT}: run this notebook on a GPU with TRAIN = True "
                       "(Colab: Runtime > Change runtime type > GPU), or train with `sctrange-train --arm`")
per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUTPUT_DIR / "s4_per_seed.csv", index=False)
numeric = [c for c in per_seed.columns if c not in ("case_id", "region", "center", "seed", "source")]
arms_by_seed = per_seed.groupby("seed")["source"].agg(set)
VERDICT_SEEDS = tuple(int(seed) for seed, arms in arms_by_seed.items() if set(ARM_ORDER) <= arms)
PRELIMINARY = set(VERDICT_SEEDS) != set(PREREGISTERED_SEEDS)
verdict_rows = per_seed[per_seed["seed"].isin(VERDICT_SEEDS)] if VERDICT_SEEDS else per_seed
diagnostics = verdict_rows.groupby(["case_id", "region", "center", "source"], as_index=False)[numeric].mean()
diagnostics.to_csv(OUTPUT_DIR / "s4_diagnostics.csv", index=False)
risk = pd.DataFrame(risk_rows)
print(f"{diagnostics['case_id'].nunique()} cases · arms {sorted(diagnostics['source'].unique())} · "
      f"{len(per_seed)} evaluations over {len(SEEDS)} seed(s) · verdict on seed(s) "
      f"{', '.join(map(str, VERDICT_SEEDS)) or 'none complete'}{' (preliminary)' if PRELIMINARY else ''}")

columns = ["mae_hu", "target_mean_signed_mm", "target_rms_mm", "target_p95_mm", "target_cvar_mm", "target_coherence",
           "target_global_ratio"]
order = [arm for arm in ARM_ORDER if arm in set(diagnostics["source"])]
display(diagnostics.groupby("source")[columns].median().reindex(order).round(3))

### 4 · Image error versus range-error tail

Figure 4 of the plan: one point per arm at the median over cases, whiskers spanning the interquartile range. Lower left
is better. The theory predicts that R3 improves on R1 mostly on the signed bias and the tail, not on MAE, and that any
gain of R5 or R6 shows on the vertical axis.

In [ ]:
grouped = diagnostics.groupby("source")
pareto = pd.concat({f"{metric}{suffix}": grouped[metric].quantile(q)
                    for metric in ("mae_hu", "target_cvar_mm")
                    for suffix, q in (("", 0.5), ("_q25", 0.25), ("_q75", 0.75))}, axis=1).reindex(order)
plot_arm_pareto(pareto).savefig(FIGURE_DIR / "fig4_pareto.png", bbox_inches="tight")
plt.show()
display(pareto.round(3))
display(diagnostics.pivot_table(index="region", columns="source", values="target_mean_signed_mm", aggfunc="median")
        .reindex(columns=order).round(3).rename_axis(columns="median signed ΔWET (mm)"))

### 5 · Pre-registered decision and secondary tests

The verdict applies `configs/s4_preregistration.json` unchanged. Secondary tests are reported whatever the verdict:
Proposition 5 on patients (signed abdominal range bias of R1 above R3, one-sided), and the MAE cost of each arm relative
to R3 (two-sided), all Holm-adjusted within this family.

In [ ]:
decision = {**decide_arms(diagnostics, prereg), "seeds": list(VERDICT_SEEDS), "preliminary": PRELIMINARY}
display(pd.Series({k: v for k, v in decision.items() if k not in ("arms", "raw_p_values")}, name="value").to_frame())
display(pd.DataFrame(decision["arms"]).T.round(4))

secondary = {"signed bias AB, R1 − R3 > 0": paired_test(diagnostics, "target_mean_signed_mm", "R1", "R3", "AB", "greater")}
for arm in ("R1", "R2", "R4", "R5", "R6"):
    secondary[f"MAE {arm} − R3"] = paired_test(diagnostics, "mae_hu", arm, "R3")
adjusted = holm({name: result["p_value"] for name, result in secondary.items()})
secondary_table = pd.DataFrame(secondary).T.assign(p_holm=pd.Series(adjusted))
display(secondary_table.round(4))

### 6 · Calibration of the R6 risk head

A calibrated head covers 90 % of rays with its VaR90 (`risk_coverage` ≈ 0.9), and the mean |ΔWET| beyond the predicted
VaR matches the predicted ES (`risk_realised_es_mm` ≈ `risk_mean_es_mm`). `risk_spearman` is the rank correlation
between predicted VaR and realised |ΔWET| over rays: the head is only useful for failure detection if it is positive.

In [ ]:
risk_columns = ["risk_rays", "risk_coverage", "risk_mean_var_mm", "risk_mean_es_mm", "risk_realised_es_mm", "risk_fz",
                "risk_spearman"]
if risk.empty:
    print("no risk maps found (R6 not trained yet)")
else:
    risk.to_csv(OUTPUT_DIR / "s4_risk.csv", index=False)
    calibration = risk.groupby("region")[risk_columns].median()
    calibration.loc["all"] = risk[risk_columns].median()
    display(calibration.round(3))

### 7 · Report

In [ ]:
report = {
    "task": TASK,
    "seeds": list(SEEDS),
    "folds": N_FOLDS,
    "split_sha256": split_hash(split),
    "overrides": OVERRIDES,
    "decision": decision,
    "secondary": secondary_table.to_dict(orient="index"),
    "medians_by_arm": diagnostics.groupby("source")[columns].median().to_dict(orient="index"),
    "risk_calibration": {} if risk.empty else risk[risk_columns].median().to_dict(),
    "missing_runs": missing,
}
(OUTPUT_DIR / "s4_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Pre-registration hashed", prereg.fingerprint[:12], True),
    ("All arms and seeds present", ", ".join(missing) or "complete", not missing),
    ("R6 risk maps evaluated", f"{len(risk)} evaluations", not risk.empty),
    ("Verdict on real data", decision["verdict"] + (f" (preliminary, seeds {VERDICT_SEEDS})" if PRELIMINARY else ""),
     decision["verdict"] != "NOT EVALUATED" and not PRELIMINARY),
], columns=["item", "detail", "passed"])
display(status)

### What each verdict implies for S6 onwards

| Verdict | Next |
|---|---|
| GAIN | Dosimetry on the 20-case subset (S6) for R1, R3, R4, R5, R6; the γ-proton test completes go/no-go 2. |
| BIAS CORRECTION SUFFICES | The paper becomes "the right estimator suffices": theory (Results A–C), R1 vs R3 vs R4, tails via R6 and the generative arm. |
| NO GAIN | Range loss dropped as a contribution; the paper rests on estimator choice, tails and projected CRPS (Results B–C). |

Whatever the verdict, the generative arm (S7–S8) is compared with R6 on tails and with R3 on the mean, where the theory
predicts equality at best.

In [ ]:
PIPELINE.finish("S4", complete=bool(not missing and decision["verdict"] != "NOT EVALUATED"), note=str(decision["verdict"]))


# S6 · First real dosimetry and go/no-go 2

Week 6 turns the range proxy of weeks 1–5 into dose, on a frozen subset of 20 development cases:

| Step | What | Plan item |
|---|---|---|
| 1 | IMPT (3 beams) and IMRT (7 beams) plans optimised on the reference CT with pyRadPlan | proton (and photon) plans on 20 cases |
| 2 | the same weights recomputed on the bulk-density baseline and on R1–R6 (every seed) | recomputation on R1–R6 |
| 3 | gamma pass rates, target DVH points, distal R80 shift of every proton beam along every ray | |
| 4 | WET error to the distal target edge against the gamma failure rate (cases) and against the R80 shift (rays) | validation of the proxy |
| 5 | the S4 range verdict combined with the proton gamma test, rule fixed in `configs/s6_preregistration.json` | **go/no-go 2** |

SynthRAD2025 releases no PTV, so the target is the 25 mm sphere at the body centroid used since week 1, and voxels
outside the body mask are set to air in the reference CT and in every sCT: only errors inside the patient reach the
dose. pyRadPlan is the dose engine of the SynthRAD2025 evaluation; its default HU → SPR table is the one used by the
range projector, so the proxy and the dose see the same stopping powers.

The verdict needs the S4–S5 runs (`outputs/s4/runs/seed<k>/<arm>`) and their report (`outputs/s4/s4_report.json`).
Dose calculation runs on the CPU (no GPU needed); each case takes several minutes per source, so the 20 cases are
resumable: every finished case is stored and reused.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead).

*Stage notebook: `notebooks/s6_dosimetry.ipynb`.*


In [ ]:
PIPELINE.begin("S6", {'task': TASK, 'cases': DOSE_CASES, 'seeds': SEED_LIST, 'workers': WORKERS}, PIPELINE_OUTPUTS / "s6" / "s6_report.json", after=('S4',))


In [ ]:
import json
import os
import warnings
from dataclasses import asdict

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sctrange.diagnostics import paired_test
from sctrange.dosimetry import (DoseConfig, average_seeds, decide_go_no_go_2, engine_checks, load_selection,
                                proxy_validation, run_dosimetry, select_cases, split_source, write_selection)
from sctrange.download import ensure_dataset, usable_records
from sctrange.data import discover_cases
from sctrange.plotting import apply_style, plot_dose_slices, plot_gamma_by_source, plot_proxy_validation
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_split, split_hash

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
DOSE_CASES = int(os.environ.get("SCT_DOSE_CASES", DOSE_CASES))
SEEDS = tuple(int(s) for s in os.environ.get("SCT_SEEDS", SEED_LIST).split(","))
WORKERS = int(os.environ.get("SCT_WORKERS", WORKERS))

OUTPUT_DIR = PATHS.output_root / "s6"
FIGURE_DIR = OUTPUT_DIR / "figures"
S4_RUNS = Path(os.environ.get("SCT_S4_RUNS_ROOT", PATHS.output_root / "s4" / "runs"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s6_preregistration.json"

prereg_bytes = PREREGISTRATION_PATH.read_bytes()
PREREG = json.loads(prereg_bytes)
PREREG_SHA256 = __import__("hashlib").sha256(prereg_bytes).hexdigest()
DOSE = DoseConfig.from_preregistration(PREREG)
ARMS = tuple(PREREG["arms"])
ARM_ORDER = ("bulk density",) + ARMS

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
print(f"task {TASK} · cases {DOSE_CASES} · seeds {SEEDS} · workers {WORKERS} · S4 runs {S4_RUNS}")
if (DOSE_CASES, SEEDS) != (PREREG["cases"]["n_cases"], tuple(PREREG["seeds"])):
    print("non-default cases or seeds: quick pass only, the verdict is not the pre-registered one")

### 1 · Frozen split, pre-registration and dosimetry subset

The split of week 1 is verified; test cases are never loaded. The 20 cases are drawn once from the development pool
(region-stratified, hashed order within region, cases with an out-of-fold R3 prediction), written with their SHA-256
and reused on every run: a rerun that would select different cases stops instead of silently reselecting.

In [ ]:
if not FROZEN_SPLIT_PATH.exists():
    raise FileNotFoundError(f"{FROZEN_SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(FROZEN_SPLIT_PATH)
development_ids = set(split["development"])
assert not development_ids & set(split["test"])
development, excluded = usable_records(DATA_ROOT, [r for r in discover_cases(DATA_ROOT)
                                                  if r.case_id in development_ids and r.task == TASK])

reference_prediction = lambda r: S4_RUNS / "seed0" / PREREG["reference_arm"] / "predictions" / f"{r.case_id}.mha"
if not any(reference_prediction(r).exists() for r in development):
    raise RuntimeError(f"no out-of-fold {PREREG['reference_arm']} prediction under {S4_RUNS}: run S4–S5 first "
                       "(notebooks/s4_regression_arms.ipynb)")
selection_path = OUTPUT_DIR / "dosimetry_cases.json"
selection_meta = {"task": TASK, "seed": PREREG["cases"]["seed"], "n_cases": DOSE_CASES, "split_sha256": split_hash(split)}
chosen_ids = select_cases(development, DOSE_CASES, PREREG["cases"]["seed"], lambda r: reference_prediction(r).exists())
selection_sha256 = write_selection(chosen_ids, selection_path, selection_meta)
chosen = [r for r in development if r.case_id in set(load_selection(selection_path)["case_ids"])]

display(pd.Series({"split sha256": split_hash(split), "pre-registration sha256": PREREG_SHA256,
                   "dosimetry selection sha256": selection_sha256, "cases": len(chosen)}, name="value").to_frame())
display(pd.Series([r.region for r in chosen]).value_counts().rename("cases").rename_axis("region").to_frame().T)

### 2 · Dose-engine checks

Two checks on a water cylinder, before any patient:

- **beam direction**: a single-beam proton plan at each plan angle must deposit dose upstream of the target along the
  projector direction and almost none downstream. This validates the angle conversion
  `gantry = (360 − angle) mod 360` between the projector (angle 90 travels along +x) and pyRadPlan (gantry 90 travels
  along −x), without which the proxy and the dose would look along different rays;
- **identity**: recomputing the optimised weights on the planning CT reproduces the reference dose (γ 100 %).

In [ ]:
checks = engine_checks(DOSE)
display(checks.round(3))
assert checks["direction_ok"].all() and (checks["identity_gamma_2_2"] >= 99.99).all(), "dose-engine check failed"

### 3 · Plans and recomputation

For each case: optimise both plans on the reference CT, recompute them on the reference CT (the reference dose, same
engine path as every sCT) and on every source. Per source and modality: γ 2 %/2 mm and 1 %/1 mm (global, 10 % cutoff),
target mean, D95 and D2 differences (% of the 60 Gy prescription), mean absolute dose difference above the cutoff, and
the range proxy on the proton beams (WET error from entry to the distal target edge). Metrics are then averaged over
seeds per case and arm, as in S4. Every case is written as soon as it is dosed, so an interrupted session resumes with
the cases that remain.

In [ ]:
per_seed, rays = run_dosimetry(chosen, S4_RUNS, OUTPUT_DIR, DOSE, ARMS, SEEDS, workers=WORKERS)
per_seed.to_csv(OUTPUT_DIR / "s6_dose_per_seed.csv", index=False)
rays.to_csv(OUTPUT_DIR / "s6_rays.csv", index=False)
cases = average_seeds(per_seed)
cases.to_csv(OUTPUT_DIR / "s6_dose.csv", index=False)

order = [s for s in ARM_ORDER if s in set(cases["source"])]
columns = ["mae_hu", "proxy_rms_mm", "proxy_cvar_mm", "proton_gamma_2_2", "proton_gamma_1_1", "photon_gamma_2_2",
           "photon_gamma_1_1", "proton_target_d95_diff_pct", "proton_dose_mae_pct"]
print(f"{cases['case_id'].nunique()} cases · sources {order} · {len(rays)} proton rays")
display(cases.groupby("source")[columns].median().reindex(order).round(3))

Central target slice of the first case: reference proton dose, then the dose difference of a few sources. Range errors
show as distal shifts behind the target; the photon plan of the same case barely changes.

In [ ]:
example = chosen[0].case_id
slices = dict(np.load(OUTPUT_DIR / "slices" / f"{example}.npz"))
shown = [s for s in ("bulk density", "R1@0", "R3@0", "R5@0") if f"proton/{s}" in slices]
for modality in ("proton", "photon"):
    plot_dose_slices(slices, shown, modality, DOSE.prescription_gy).savefig(
        FIGURE_DIR / f"dose_{modality}_{example}.png", bbox_inches="tight")
    plt.show()
plot_gamma_by_source(cases, order).savefig(FIGURE_DIR / "gamma_by_source.png", bbox_inches="tight")
plt.show()

### 4 · Does the range proxy track dose?

Pre-registered criteria (`proxy` block): across cases and sources, the RMS WET error to the distal target edge on the
plan beams correlates with the proton γ 2 %/2 mm failure rate (Spearman ≥ 0.5, case-bootstrap 95 % interval above 0)
**and still does when MAE is held fixed** (partial Spearman, interval above 0); per ray, the WET error predicts the
distal R80 shift (Spearman between ΔWET and −ΔR80 ≥ 0.7). The Theil–Sen slope of ΔR80 on ΔWET is expected near
−1/SPR at the distal edge (about −1 in soft tissue). The MAE-only Spearman is shown for contrast, and the case Spearman without the bulk-density baseline (`arms_only_case_spearman`, secondary: the
baseline is far from every arm and can carry the pooled correlation on its own).

In [ ]:
validation = proxy_validation(cases, rays, PREREG, n_boot=2000, seed=0)
display(pd.Series(validation, name="value").to_frame())
plot_proxy_validation(cases, rays).savefig(FIGURE_DIR / "proxy_validation.png", bbox_inches="tight")
plt.show()
if not rays.empty:
    by_source = split_source(rays).groupby("arm").apply(
        lambda f: pd.Series({"rays": len(f), "median ΔWET (mm)": f["delta_wet_mm"].median(),
                             "median ΔR80 (mm)": f["delta_r80_mm"].median(),
                             "Spearman(ΔWET, −ΔR80)": f["delta_wet_mm"].corr(-f["delta_r80_mm"], method="spearman")}),
        include_groups=False)
    display(by_source.reindex([s for s in ARM_ORDER if s in by_source.index]).round(3))

### 5 · Go/no-go 2

The rule of `configs/s6_preregistration.json`, applied unchanged:

| Verdict | Condition |
|---|---|
| GO | a method arm beats R3 **and** R4 on target CVaR90 in S4 (MAE condition met) **and** beats R3 on proton γ 2 %/2 mm here |
| RANGE GAIN WITHOUT DOSE CONFIRMATION | it beats R3 and R4 in S4, but no method arm beats R3 on proton γ |
| RIGHT ESTIMATOR SUFFICES | some method arm beats R3 in S4, none beats R4: the gain is a tissue-level bias |
| NO-GO | no method arm beats R3 in S4 |

γ tests are one-sided paired Wilcoxon over the dosimetry cases, Holm over the four comparisons. Secondary results are
reported whatever the verdict.

In [ ]:
s4_report_path = PATHS.output_root / PREREG["s4_report"]
s4_report = json.loads(s4_report_path.read_text()) if s4_report_path.exists() else None
if s4_report is None:
    print(f"{s4_report_path} not found: run notebooks/s4_regression_arms.ipynb first")
decision = decide_go_no_go_2(cases, s4_report, PREREG)
display(pd.Series({k: v for k, v in decision.items() if k not in ("arms", "raw_p_values")}, name="value").to_frame())
display(pd.DataFrame(decision["arms"]).T)

secondary = {
    "proton γ 2/2, R1 − R3": paired_test(cases, "proton_gamma_2_2", "R1", "R3"),
    "proton γ 2/2, R4 − R3": paired_test(cases, "proton_gamma_2_2", "R4", "R3"),
    "photon γ 2/2, R5 − R3": paired_test(cases, "photon_gamma_2_2", "R5", "R3"),
}
display(pd.DataFrame(secondary).T.round(4))
dvh = ["proton_target_mean_diff_pct", "proton_target_d95_diff_pct", "proton_target_d2_diff_pct",
       "photon_target_d95_diff_pct"]
display(cases.pivot_table(index="region", columns="source", values="proton_target_d95_diff_pct", aggfunc="median")
        .reindex(columns=order).round(2).rename_axis(columns="median proton D95 difference (% of prescription)"))
display(cases.groupby("source")[dvh].median().reindex(order).round(2))

### 6 · Report

In [ ]:
report = {
    "task": TASK,
    "seeds": list(SEEDS),
    "split_sha256": split_hash(split),
    "preregistration_sha256": PREREG_SHA256,
    "selection_sha256": selection_sha256,
    "cases": [r.case_id for r in chosen],
    "dose_config": asdict(DOSE),
    "engine_checks": checks.to_dict(orient="records"),
    "proxy_validation": validation,
    "decision": decision,
    "secondary": secondary,
    "medians_by_source": cases.groupby("source")[columns].median().to_dict(orient="index"),
}
(OUTPUT_DIR / "s6_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Dosimetry subset frozen", selection_sha256[:12], True),
    ("Dose-engine checks", f"{len(checks)} beams", bool(checks["direction_ok"].all())),
    ("All arms and seeds dosed", f"{per_seed['source'].nunique()} sources",
     all(f"{a}@{s}" in set(per_seed["source"]) for a in ARMS for s in SEEDS)),
    ("Proxy validation", validation["verdict"], validation["verdict"] == "VALIDATED"),
    ("Go/no-go 2", decision["verdict"], decision["verdict"] != "NOT EVALUATED"),
], columns=["item", "detail", "passed"])
display(status)

### What each verdict implies

| Verdict | Next |
|---|---|
| GO | The paper keeps the range-aware loss as its method contribution; S7–S8 compare the generative arm with R6 on tails. |
| RANGE GAIN WITHOUT DOSE CONFIRMATION | Report the range gain with the γ effect size and its interval; 20 cases may be underpowered for γ, so the S11 test set (opened once) decides. |
| RIGHT ESTIMATOR SUFFICES | The paper becomes "the right estimator suffices": theory (Results A–C), R1 vs R3 vs R4, tails via R6 and the generative arm. |
| NO-GO | The range loss is dropped as a contribution; the paper rests on estimator choice, tails and projected CRPS. |

Whatever the verdict, a **VALIDATED** proxy lets the later weeks (S9–S10 ablations, test set) report range metrics
without recomputing dose for every variant; **PARTIAL** or **NOT VALIDATED** means dose must be recomputed for the
final comparison.

In [ ]:
PIPELINE.finish("S6", complete=bool(decision["verdict"] != "NOT EVALUATED" and all(f"{a}@{s}" in set(per_seed["source"]) for a in ARMS for s in SEEDS)), note=str(decision["verdict"]))


# S7 · Generative arm: stochastic-interpolant flow matching

Weeks 7–8 of the plan build a generative sCT model with the backbone and budget of the regression arms and ask what it
adds. Every item of that block runs here:

| Plan item | Section |
|---|---|
| FM source → CT with a stochastic interpolant, same backbone; sweep η ∈ {0, 0.05, 0.1, 0.2} | 2, 3 |
| quality against the number of steps (1, 2, 4, 8, 16, 32) | 4 |
| projected CRPS, coverage, uncertainty–error correlation | 5 |
| generative vs the R6 head on tails; generative vs R3 on the mean (the theory predicts equality at best) | 6 |

**Model** (`sctrange.generative`, `configs/s7_preregistration.json`). In encoded CT units,
x_t = (1 − t) x₀ + t x₁ + γ(t) z with γ(t) = 2η t(1 − t), x₀ the bulk-density image (water in the body), x₁ the
reference CT and z Gaussian noise inside the body. The U-Net of R1–R6, with FiLM time conditioning, predicts the drift
and, for η > 0, the noise; samples come from Euler–Maruyama from x₀. With η = 0 sampling is deterministic and the
ensemble collapses to one image, and a single step returns the conditional mean in HU: the sweep shows what
the stochastic part buys.

**Point prediction.** The SPR mean of the samples, g⁻¹(mean g(sample)), is the Bayes act of the squared range loss
(Proposition 4); it is evaluated with the S4 metrics. Ensembles are scored on the WET error of every ray from entry to
the distal target edge: projected CRPS (proper, not strictly proper on finite segment families, Proposition 8),
coverage of central 90 % intervals, PIT, and the correlation between ensemble spread and the error of the mean.

**Compute.** 4 generators × 5 folds × 30 000 steps (about as much as one S4 seed), then sampling: K samples × N steps
per slice. Sampling is the costly part, so the pre-registration fixes nested case subsets (30 for the sweep, 10 for the
step curve, 100 for the main comparison). Every fold checkpoint and every evaluated case is stored and reused.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead).

*Stage notebook: `notebooks/s7_generative.ipynb`.*


In [ ]:
PIPELINE.begin("S7", {'task': TASK, 'steps': STEPS, 'folds': FOLDS, 'etas': ETAS, 'sweep': SWEEP_CASES, 'step_curve': STEP_CURVE_CASES, 'main': MAIN_CASES, 'train': TRAIN}, PIPELINE_OUTPUTS / "s7" / "s7_report.json", after=('S4',))


In [ ]:
import hashlib
import json
import os
import warnings
from dataclasses import asdict, replace

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.data import discover_cases, load_case, read_prediction, read_volume
from sctrange.download import ensure_dataset, usable_records
from sctrange.runs import run_complete
from sctrange.evaluation import DiagnosticConfig, evaluate_case, evaluate_risk
from sctrange.generative import (GenerativeConfig, _load_generator, comparison_frame, crossfit_generative,
                                 collapse_ratio, decide_generative, evaluate_generative, run_name, sample_case,
                                 select_eta, spr_mean)
from sctrange.hlut import HLUT
from sctrange.plotting import (apply_style, plot_eta_sweep, plot_paired_differences, plot_pit, plot_sample_gallery,
                               plot_steps_curve, plot_training_curves)
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_selection, load_split, select_cases, split_hash, write_selection
from sctrange.training import visible_devices

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
TRAIN = os.environ.get("SCT_TRAIN_IN_NOTEBOOK", "1" if TRAIN else "0") == "1"
STEPS = int(os.environ.get("SCT_STEPS", STEPS))
N_FOLDS = int(os.environ.get("SCT_FOLDS", FOLDS))
ETAS = tuple(float(e) for e in os.environ.get("SCT_ETAS", ETAS).split(","))
SWEEP_CASES = int(os.environ.get("SCT_SWEEP_CASES", SWEEP_CASES))
STEP_CURVE_CASES = int(os.environ.get("SCT_STEP_CASES", STEP_CURVE_CASES))
MAIN_CASES = int(os.environ.get("SCT_MAIN_CASES", MAIN_CASES))

OUTPUT_DIR = PATHS.output_root / "s7"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_GEN_RUNS_ROOT", OUTPUT_DIR / "runs"))
S4_RUNS = Path(os.environ.get("SCT_S4_RUNS_ROOT", PATHS.output_root / "s4" / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", PATHS.cache_root / "cache"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s7_preregistration.json"

prereg_bytes = PREREGISTRATION_PATH.read_bytes()
PREREG = json.loads(prereg_bytes)
PREREG_SHA256 = hashlib.sha256(prereg_bytes).hexdigest()
SEED = 20261013
DEVICES = visible_devices()
DEVICE = DEVICES[0]
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
BASE_CONFIG = GenerativeConfig(steps=STEPS, seed=PREREG["seed"], device=DEVICE)
hlut = HLUT.pyradplan_default()

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
TRAIN_IN_NOTEBOOK = TRAIN and (DEVICE.startswith("cuda") or os.environ.get("SCT_TRAIN_ON_CPU") == "1")
if TRAIN and not TRAIN_IN_NOTEBOOK:
    print("no GPU: training skipped, existing checkpoints only (SCT_TRAIN_ON_CPU=1 forces CPU training)")
torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
defaults = (PREREG["steps"], PREREG["folds"], tuple(PREREG["eta_grid"]), PREREG["sweep"]["cases"],
            PREREG["steps_curve"]["cases"], PREREG["main"]["cases"])
if (STEPS, N_FOLDS, ETAS, SWEEP_CASES, STEP_CURVE_CASES, MAIN_CASES) != defaults:
    print("non-default budget: quick pass only, the verdict is not the pre-registered one")
print(f"task {TASK} · devices {', '.join(DEVICES)} · eta {ETAS} · steps {STEPS} · folds {N_FOLDS}")

### 1 · Frozen split, pre-registration and case subsets

The three evaluation subsets are drawn once with the pre-registered seed (region-stratified, hashed order within
region) from the development cases, written with their SHA-256 under `outputs/s7/subsets/` and reused on every run: a
rerun that would select other cases stops instead of silently reselecting. Every case has seed-0 R3 and R6 predictions
once S4 is finished, so every comparison is paired. Because
the order within a region is fixed, the smaller subsets are, up to the allocation by region, the first cases of the
larger ones.

In [ ]:
if not FROZEN_SPLIT_PATH.exists():
    raise FileNotFoundError(f"{FROZEN_SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(FROZEN_SPLIT_PATH)
development_ids = set(split["development"])
assert not development_ids & set(split["test"])
development, excluded = usable_records(DATA_ROOT, [r for r in discover_cases(DATA_ROOT)
                                                  if r.case_id in development_ids and r.task == TASK])

prediction = lambda arm, r: S4_RUNS / "seed0" / arm / "predictions" / f"{r.case_id}.mha"
HAVE_S4 = all(run_complete(S4_RUNS / "seed0" / arm, [r.case_id for r in development]) for arm in ("R3", "R6"))
if not HAVE_S4:
    print(f"seed-0 R3/R6 runs not finished under {S4_RUNS}: the generator is evaluated alone, section 6 is skipped")
seed = PREREG["subsets"]["seed"]
subset_ids = {}
for name, n in (("sweep", SWEEP_CASES), ("steps", STEP_CURVE_CASES), ("main", MAIN_CASES)):
    path = OUTPUT_DIR / "subsets" / f"{name}.json"
    meta = {"task": TASK, "seed": seed, "n_cases": n, "split_sha256": split_hash(split)}
    write_selection(select_cases(development, n, seed), path, meta)  # refuses to change a frozen subset
    subset_ids[name] = load_selection(path)["case_ids"]
subsets = {name: [r for r in development if r.case_id in set(ids)] for name, ids in subset_ids.items()}
display(pd.Series({"split sha256": split_hash(split), "pre-registration sha256": PREREG_SHA256,
                   "development cases": len(development), **{f"{k} cases": len(v) for k, v in subsets.items()}},
                  name="value").to_frame())

### 2 · Cross-fitted training, one generator per η

Each generator is cross-fitted on the whole development pool of the task (5 stratified folds, seed 0), so that every
evaluated case is sampled by a model that never saw it. Checkpoints are reused when their configuration and training cases
match and refused otherwise; a fold cut mid-training resumes from its last in-fold save. Sampled cases are cached under
`runs/<eta>/eval/`, so the sections below also resume after an interruption. The losses are the drift and denoiser regressions, averaged over body pixels.

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")
configs = {eta: replace(BASE_CONFIG, eta=eta) for eta in ETAS}
histories, missing = {}, []
for eta, config in configs.items():
    run_dir = RUNS_ROOT / run_name(config)
    if TRAIN_IN_NOTEBOOK:
        crossfit_generative(development, cache, config, N_FOLDS, run_dir, DEVICES)
    if not all((run_dir / f"fold{k}.pt").exists() for k in range(N_FOLDS)):
        missing.append(run_name(config))
    elif (run_dir / "history.csv").exists():
        histories[f"η = {eta:g}"] = pd.read_csv(run_dir / "history.csv")
if missing:
    raise RuntimeError(f"missing generators: {', '.join(missing)}; run this notebook on a GPU with TRAIN = True, "
                       "or `sctrange-generative --eta ...`")
display(pd.Series(asdict(BASE_CONFIG), name="configuration").to_frame())
plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

### 3 · η sweep and pre-registered choice of η

Every generator samples the sweep cases (K = 8, N = 16). Among the stochastic generators (η > 0), the selected η
minimises the median projected CRPS (ties go to the smaller η) and is used in every later section. η = 0 is the
control: it is deterministic, so its ensemble spread must be at most 5 % of the smallest spread among η > 0 (collapse
check), and its CRPS equals the absolute WET error of its single image.

In [ ]:
sweep_spec = PREREG["sweep"]
sweep_frames, sweep_pits = [], {}
for eta, config in configs.items():
    rows, pits = evaluate_generative(subsets["sweep"], cache, RUNS_ROOT / run_name(config), config, hlut, DIAGNOSTIC,
                                     sweep_spec["n_samples"], sweep_spec["n_steps"], DEVICE, full=False)
    sweep_frames.append(rows)
    sweep_pits[eta] = pits
sweep = pd.concat(sweep_frames, ignore_index=True)
sweep.to_csv(OUTPUT_DIR / "s7_sweep.csv", index=False)
SELECTED_ETA = select_eta(sweep, sweep_spec["selection_metric"])
summary = sweep.groupby("eta")[["mae_hu", "target_rms_mm", "pcrps_mm", "pcrps_point_mm", "coverage90", "spread_mm",
                                "unc_err_spearman"]].median()
display(summary.round(3))
COLLAPSE_RATIO = collapse_ratio(sweep)
COLLAPSES = bool(np.isfinite(COLLAPSE_RATIO) and COLLAPSE_RATIO <= PREREG["collapse"]["max_spread_ratio"])
print(f"selected eta = {SELECTED_ETA:g} · spread ratio eta 0 / eta > 0 = {COLLAPSE_RATIO:.3g} "
      f"({'collapses' if COLLAPSES else 'does not collapse'})")
plot_eta_sweep(sweep, SELECTED_ETA).savefig(FIGURE_DIR / "eta_sweep.png", bbox_inches="tight")
plt.show()

### 4 · Quality against the number of steps

The selected generator samples the step-curve cases with 1, 2, 4, 8, 16 and 32 steps (K = 8). With one step every
sample is the conditional mean in HU, so the curve runs from a regression-like estimate (left) to samples of the
learned conditional law (right).

In [ ]:
curve_spec = PREREG["steps_curve"]
selected = configs[SELECTED_ETA]
curve = pd.concat([evaluate_generative(subsets["steps"], cache, RUNS_ROOT / run_name(selected), selected, hlut,
                                       DIAGNOSTIC, curve_spec["n_samples"], n, DEVICE, full=False)[0]
                   for n in curve_spec["n_steps"]], ignore_index=True)
curve.to_csv(OUTPUT_DIR / "s7_steps_curve.csv", index=False)
display(curve.groupby("n_steps")[["mae_hu", "sample_mae_hu", "target_rms_mm", "pcrps_mm", "coverage90",
                                  "spread_mm"]].median().round(3))
plot_steps_curve(curve).savefig(FIGURE_DIR / "steps_curve.png", bbox_inches="tight")
plt.show()

### 5 · Main evaluation: calibration of the range-error distribution

The selected generator samples the main cases (K = 16, N = 16). Its SPR-mean prediction is written to
`runs/<eta>/predictions/` (it can be dosed like an S4 arm) and evaluated with the S4 metrics, permutation controls
included. A calibrated ensemble gives a flat PIT histogram, coverage near 0.9 and a positive spread–error correlation.

In [ ]:
main_spec = PREREG["main"]
main_rows, main_pits = evaluate_generative(subsets["main"], cache, RUNS_ROOT / run_name(selected), selected, hlut,
                                           DIAGNOSTIC, main_spec["n_samples"], main_spec["n_steps"], DEVICE,
                                           write_mean=True, full=True)
main_rows.to_csv(OUTPUT_DIR / "s7_main.csv", index=False)
calibration = main_rows.groupby("region")[["coverage90", "spread_mm", "pcrps_mm", "pcrps_point_mm",
                                           "unc_err_spearman", "risk_coverage"]].median()
calibration.loc["all"] = main_rows[calibration.columns].median()
display(calibration.round(3))
print(f"cases with positive spread–error correlation: {(main_rows['unc_err_spearman'] > 0).mean():.0%}")
plot_pit(main_pits).savefig(FIGURE_DIR / "pit.png", bbox_inches="tight")
plt.show()

example = subsets["main"][0]
folds = pd.read_csv(RUNS_ROOT / run_name(selected) / "folds.csv", dtype={"case_id": str}).set_index("case_id")["fold"]
model = _load_generator(RUNS_ROOT / run_name(selected) / f"fold{folds[example.case_id]}.pt", replace(selected, device=DEVICE))
case = cache.load(example.case_id)
gallery = sample_case(model, case, selected, 4, main_spec["n_steps"], DEVICE, seed=1)
centre = int(np.argmax(np.asarray(case.mask).sum(axis=(1, 2))))
plot_sample_gallery(np.asarray(case.ct, dtype=np.float32), gallery, spr_mean(gallery, hlut), np.asarray(case.mask),
                    centre).savefig(FIGURE_DIR / "samples.png", bbox_inches="tight")
plt.show()

### 6 · Generative arm against R3 (mean) and R6 (tails)

R3 and R6 (seed 0, out of fold) are evaluated on the same main cases with the S4 metrics and, for R6, its risk maps.
A point prediction is a degenerate ensemble, so its projected CRPS is its mean absolute WET error. Pre-registered
tests (Holm over the four):

| Test | Column | Alternative | Theory |
|---|---|---|---|
| mean better than R3 | RMS ΔWET of the point prediction | G < R3 | not expected: equality at best |
| proper score better than R3 | projected CRPS | G < R3 | expected if the spread is informative |
| tail score vs R6 | FZ0 score of (VaR90, ES90) of the point prediction's \|ΔWET\| | two-sided | open |
| tail calibration vs R6 | \|coverage of VaR90 − 0.9\| | two-sided | open |

Each model forecasts the tail of its own error, so the FZ0 scores of G and R6 concern different outcomes; the
calibration test compares them on a common scale.

In [ ]:
if HAVE_S4:
    reference_path = OUTPUT_DIR / "s7_reference_arms.csv"
    risk_path = OUTPUT_DIR / "s7_reference_risk.csv"
    done = pd.read_csv(reference_path) if reference_path.exists() else pd.DataFrame(columns=["case_id", "source"])
    risk_done = pd.read_csv(risk_path) if risk_path.exists() else pd.DataFrame(columns=["case_id", "source"])
    ref_rows, risk_rows = [done], [risk_done]
    for record in subsets["main"]:
        if record.case_id in set(done["case_id"]):
            continue
        volumes = load_case(record, ("ct", "mask"))
        ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
        keys = {"case_id": record.case_id, "region": record.region, "center": record.center}
        rows = []
        for arm in ("R3", "R6"):
            sct = read_prediction(prediction(arm, record), ct.shape)
            rows.append({**keys, "source": arm, **evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)})
        run = S4_RUNS / "seed0" / "R6" / "risk"
        if (run / f"{record.case_id}_var.mha").exists():
            var = read_volume(run / f"{record.case_id}_var.mha").array
            excess = read_volume(run / f"{record.case_id}_excess.mha").array
            sct = read_prediction(prediction("R6", record), ct.shape)
            risk_rows.append(pd.DataFrame([{**keys, "source": "R6",
                                            **evaluate_risk(ct, mask, sct, var, excess, spacing, hlut, DIAGNOSTIC)}]))
        ref_rows.append(pd.DataFrame(rows))
    reference = pd.concat(ref_rows, ignore_index=True)
    reference.to_csv(reference_path, index=False)
    reference_risk = pd.concat(risk_rows, ignore_index=True)
    reference_risk.to_csv(risk_path, index=False)

    frame = comparison_frame(main_rows, reference, reference_risk, PREREG["generative_name"])
    frame.to_csv(OUTPUT_DIR / "s7_comparison.csv", index=False)
    decision = decide_generative(frame, PREREG, sweep)
    display(frame.groupby("source")[["mae_hu", "target_rms_mm", "target_mean_signed_mm", "target_cvar_mm", "pcrps_mm",
                                     "risk_fz", "risk_coverage", "risk_spearman"]].median().round(3))
    display(pd.DataFrame(decision["tests"]).T[["column", "first", "second", "alternative", "cases",
                                               "median_difference", "p_value", "p_holm", "significant"]])
    display(pd.Series({k: v for k, v in decision.items() if k != "tests"}, name="value").to_frame())
    plot_paired_differences(frame, [("target_rms_mm", "G", "R3", "RMS ΔWET (mm)"),
                                    ("pcrps_mm", "G", "R3", "projected CRPS (mm)"),
                                    ("risk_fz", "G", "R6", "FZ0 tail score"),
                                    ("risk_miscalibration", "G", "R6", "|VaR90 coverage − 0.9|")]).savefig(
        FIGURE_DIR / "paired_differences.png", bbox_inches="tight")
    plt.show()
else:
    decision = {"tests": {}, "note": "no S4 runs: comparison not evaluated"}

### 7 · Report

In [ ]:
report = {
    "task": TASK,
    "split_sha256": split_hash(split),
    "preregistration_sha256": PREREG_SHA256,
    "subsets": subset_ids,
    "config": asdict(BASE_CONFIG),
    "etas": list(ETAS),
    "selected_eta": SELECTED_ETA,
    "collapse_ratio": COLLAPSE_RATIO,
    "sweep_medians": summary.to_dict(orient="index"),
    "steps_curve_medians": curve.groupby("n_steps")[["mae_hu", "target_rms_mm", "pcrps_mm"]].median()
                                .to_dict(orient="index"),
    "calibration": calibration.to_dict(orient="index"),
    "decision": decision,
}
(OUTPUT_DIR / "s7_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Generators trained", ", ".join(run_name(c) for c in configs.values()), not missing),
    ("η selected (pre-registered rule)", f"{SELECTED_ETA:g}", True),
    ("η = 0 collapses", f"spread ratio {COLLAPSE_RATIO:.3g}", COLLAPSES),
    ("Steps curve", f"{curve['n_steps'].nunique()} step counts", True),
    ("Main evaluation", f"{len(main_rows)} cases", len(main_rows) > 0),
    ("Comparison with R3 and R6", "done" if HAVE_S4 else "no S4 runs", HAVE_S4),
], columns=["item", "detail", "passed"])
display(status)

### Reading the results

| Outcome | Meaning for the paper |
|---|---|
| mean not better than R3, proper score better | The theory's prediction: the generator adds a calibrated distribution, not a better point estimate. Result C (tails need the joint law) gets its empirical arm. |
| mean better than R3 | The generator is a better-fitted model, not a better estimator; check R3's capacity and training before claiming anything. |
| proper score not better than R3 | The spread is uninformative or miscalibrated (PIT shape tells which); the generative arm stays a negative result. |
| G better calibrated than R6 on tails | Tail estimation from samples beats the plan-independent density head; R6 remains the cheap option. |

The plan's S8 week continues this arm without new code: the main evaluation for task 2 (CBCT, `TASK = "2"`), the
selected generator's SPR mean dosed with the S6 notebook, and a larger K on the cases where coverage is off.

In [ ]:
PIPELINE.finish("S7", complete=bool(HAVE_S4 and not missing), note=str(f"eta {SELECTED_ETA:g}"))


## Pipeline state


In [ ]:
PIPELINE.show()
